# ИИ-Тренажер для развития навыков анализа и ведения дискуссии на русском языке на базе аудио с YouTube

(An AI-Assisted Russian Language Discussion Training Tool based on YouTube audios)

# Подключаем сессию Colab на CPU

(Select 'CPU' Colab runtime type)

# Получаем исходный аудиофрагмент

-> если у вас уже наработан аудио и/или текстовый материал, можете сразу перейти ниже к разделу 'Транскрибируем аудио...' или 'Работаем с контентом, используя ИИ...'

# Предварительная подготовка

а) Скопируйте ссылку на тот видеоролик, аудио от которого вы хотите скачать;

б) Запустите следующие три ячейки ниже.

In [ ]:

# IINSTALL THE yt-dlp PACKAGE to handle YouTube audio

!pip install -q yt-dlp -U

In [ ]:

# MOUNT YOUR GOOGLE DRIVE (to use as storage)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTS

import yt_dlp
import os
import shutil
from IPython.display import Audio    #, display
import subprocess
import textwrap

# Скачиваем аудио (англ./рус. яз.) с YouTube: подготовка

Запустите следующие две ячейки ниже.

In [ ]:
# -----------------------
# HELPER FUNCTIONS
# -----------------------


# Returns normalized topic name (specified by user) as filename base for automated file naming
def get_topic_name_from_user():
    user_topic_name_input = input("\nВведите придуманное вами 'смысловое наименование': ").strip()
    t_name_base = "".join(c for c in user_topic_name_input if c.isalnum() or c in (' ', '-', '_')).strip()
    t_name_base = t_name_base.replace(" ", "_").lower()

    return t_name_base



# Formats text output for better readability
def wrapped_print(text, width=80):
    for line in text.split('\n'):
        print('\n'.join(textwrap.wrap(line, width)))


# Returns a lightweight mp3 version of the audio to replay in the Colab notebook
def convert_wav_to_mp3(wav_path, mp3_path):

    subprocess.run([
        "ffmpeg", "-y",
        "-i", wav_path,
        "-vn",
        "-ar", "44100",
        "-ac", "1",
        "-b:a", "128k",
        mp3_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)
    return mp3_path

In [ ]:

# ---------------------
# MAIN FUNCTION
# ---------------------

def download_youtube_audio():
    output_base = "/content/YouTube_full_audio"

    while True:
        url = input("\nВставьте сюда заранее скопированную ссылку на ролик в YouTube, либо нажмите Enter, чтобы выйти: ").strip()

        if url == "":
            return "cancelled"

        # Basic URL sanity check
        if not (url.lower().startswith(("http://", "https://"))
                and ("youtube.com" in url or "youtu.be" in url)):
            print("\nВвод не похож на корректную YouTube-ссылку.")
            retry = input("Хотите попробовать ввести ссылку еще раз (да/нет)?: ").strip().lower()
            if retry != "да":
                return "cancelled"
            continue



        # Configure yt-dlp
        ydl_opts = {
            'format': 'bastaudio/best',
            'outtmpl': output_base + '.%(ext)s',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav'
            }],
            'quiet': False
        }


        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            error_code = ydl.download([url])

        if error_code != 0:
            raise RuntimeError("\nСкачать аудио не удалось. Проверьте ссылку, доступность контента и/или интернет-соединение...")

        wav_path = output_base + ".wav"

        if not os.path.exists(wav_path) or os.path.getsize(wav_path) == 0:
            raise FileNotFoundError("\nСоздать временные файлы с нужным аудио не удалось, попробуйте заново запустить скачивание.")


        mp3_path = output_base + ".mp3"

        convert_wav_to_mp3(wav_path, mp3_path)

        # User info
        wav_size = os.path.getsize(wav_path) / (1024 * 1024)
        mp3_size = os.path.getsize(mp3_path) / (1024 * 1024)
        print(f"\nСкачивание завершено, полученное аудио загружено для обработки и контрольного прослушивания.")
        print(f"Размер аудиофайла .wav (для обработки) {wav_path} —-> {wav_size:.2f} MiB")
        print(f"Размер аудиофайла .mp3 (для контрольного прослушивания) {mp3_path} —-> {mp3_size:.2f} MiB")

        return {"wav": wav_path, "mp3": mp3_path}

# Скачиваем аудио: запуск
Запустите следующую ячейку ниже, чтобы получить намеченное аудио.
Ссылка на соответствующий ролик YouTube уже должна быть скопирована!

⚠️ При сбоях автоматически будет сделано не более трех попыток скачивания подряд во избежание попадания во временный бан YouTube за слишком активное скачивание контента.

Получаемое аудио загружается в два временных файла в разных форматах: .wav - для дальнейшей обработки, и .mp3
для обеспечения возможности контрольного прослушивания и определения тайминга нужных фрагментов.

In [ ]:
# --------------------
# RUN YouTube audio download
# --------------------

MAX_FAIL_ATTEMPTS = 3
attempt = 0

# User info
print("Приступаю к скачиванию аудио...")

while True:
    try:
        result = download_youtube_audio()

        if result == "cancelled":
            print("\nСкачивание аудио отменено...")
            break

        # User info
        wrapped_print(
"\nЧтобы прослушать скачанное аудио и определить тайминги для вырезания нужного фрагмента, запустите следующую ячейку ниже."
        )
        break

    except Exception as e:
        attempt += 1
        print(f"\nАудио скачать не удалось, ошибка: {e}")

        if MAX_FAIL_ATTEMPTS and attempt >= MAX_FAIL_ATTEMPTS:
            wrapped_print("\n‼ Три попытки скачивания подряд выдали ошибку. Сделайте перерыв, проверьте возможные причины. Через некоторое время запустите всё ячейки с начала, чтобы снова попытаться скачать аудио.")
            break

        retry = input("\nХотите предпринять еще одну попытку скачать аудио? (да/нет): ").strip().lower()
        if retry != "да":
            print("\Попытки скачать аудио прекрашены")
            break

In [ ]:
#@title Контрольное прослушивание скачанного аудио

# Replay widget (uses the previously created lightweight mono mp3 file)
if isinstance(result, dict):

    display(Audio(result["mp3"]))

# Вырезаем нужную часть (фрагмент) из скачанного аудио: подготовка

а) Определитесь с таймингами фрагмента

б) Запустите следующую ячейку ниже.

Тайминги фрагмента задаются в формате ЧасЧас:МинМин:СекСек и показывают:
- время начала фрагмента (в исходном аудио) и
- общую ⚠️  длительность фрагмента => !! считая от момента его начала

In [ ]:
# -------------------
# MAIN FUNCTION
# -------------------


def cut_audio(start_time, input_file, duration, temp_output_file):

    command = [
        'ffmpeg',
        '-y',
        '-ss', str(start_time),
        '-i', input_file,
        '-t', str(duration),
        '-acodec', 'pcm_s16le',
        temp_output_file
    ]

    try:
        subprocess.run(command, check=True)
    except subprocess.CalledProcessError:
        print("\nВырезать аудиофрагмент не удалось. Проверьте формат тайминга: ЧЧ:ММ:СС.")
        return

# Вырезаем аудиофрагмент: запуск

Запустите следующую ячейку ниже, чтобы вырезать фрагмент.

Полученный аудиофрагмент тоже загружается в два временных файла - .wav для сохранения и дальнейшей обработки,
и .mp3 для обеспечения возможности контрольного прослушивания.

In [ ]:
# --------------------
# RUN audio trimming
# --------------------


# Prompt the user to specify audio fragment timing
if isinstance(result, dict):
    print("\nПриступаю к вырезанию аудиофрагмента...")

    start_u = input("\nВведите время начала фрагмента в формате ЧЧ:ММ:СС  ").strip()
    duration_u = input("Введите продолжительность фрагмента в формате ЧЧ:ММ:СС ").strip()

    input_file = result["wav"]
    preview_wav = "/content/preview_cut.wav"
    preview_mp3 = "/content/preview_cut.mp3"

    # Trim the .wav audio file
    cut_audio(start_time=start_u,
              input_file=input_file,
              duration=duration_u,
              temp_output_file=preview_wav)

    # Lightweight .mp3 version for replay in the notebook
    convert_wav_to_mp3(preview_wav, preview_mp3)

    print("\n\nАудиофрагмент вырезан и успешно загружен во временные файлы .wav  и .mp3.")

    # User info
    wrapped_print(
    """\n\nЧтобы прослушать вырезанный фрагмент, запустите следующую ячейку ниже.
\nЕсли тайминг фрагмента вас не удовлетворяет, вырежьте его заново, еще раз запустив текущую ячейку и введя скорректированные тайминги.
\nСнова запустите следующую ячейку ниже, чтобы прослушать заново вырезанный фрагмент."""
    )

In [ ]:
#@title Контрольное прослушивание вырезанного аудиофрагмента

# Replay widget (uses the lightweight mp3 version of the trimmed audio)
Audio(preview_mp3)

# Сохраняем вырезанный аудиофрагмент на Google Drive

⚠️ ОЧЕНЬ ВАЖНО ⚠️  : Придумайте и введите уникальное(!) 'смысловое наименование' - вроде темы - для каждого вырезанного аудиофрагмента, который вы сохраняете для дальнейшей работы.

На каждом этапе обработки (сохранение вырезанного аудиофрагмента, его транскрибация, при необходимости - перевод полученного текста и/или формирование основы обсуждения для
ИИ-собеседников) создаются соответствующие файлы, которые сохраняются в облачном хранилище Google Drive. Введенное вами смысловое наименование становится основной частью имени каждого файла
из этой 'цепочки', оно логически привязывает все эти файлы к исходному аудиофрагменту.
В дальнейшем при выборе файлов для обработки нужно указывать только эти смысдовые наименования, а нужный файл будет загружаться автоматически.

Сформулируйте это наименование таким образом, чтобы оно было 'говорящим' и запоминающимся для вас.

При вырезании нескольких фрагментов из одного исходного аудио (через повторный запуск соответствующей ячейки) дайте каждому фрагменту
смысловое наименование, отличающееся хотя бы одной буквой/цифрой от смысловых наименований всех других фрагментов, вырезанных из того же аудио.

⚠️ Важно !! При вводе смысловых наименований прибавлять какие-либо расширения (как, например, .txt, .wav и т.п.) НЕ НУЖНО ⚠️

In [ ]:
# -----------------
# FOLDER PATH
# -----------------

# Specify path to the folder set up in your Google Drive to save processed material from the project
DRIVE_FOLDER = "/content/drive/My Drive/...your folder name for saved files with project material..."
os.makedirs(DRIVE_FOLDER, exist_ok=True)

In [ ]:
# ------------------------
# SAVE the trimmed .wav audio file to your Google Drive for further processing
# (file name is built automatically with user-specified normalized topic name as filename base)
# ------------------------


t_name_base = get_topic_name_from_user()

final_cut_file = os.path.join(DRIVE_FOLDER, f"{t_name_base}_a_cut.wav")


confirm = input("\nВведите да, чтобы сохранить вырезанный аудиофрагмент на Google Drive для дальнейшей обработки, либо нажмите Enter, чтобы выйти: ").strip().lower()

if confirm == 'да':
    shutil.copy2(preview_wav, final_cut_file)
    wrapped_print(f"\nВырезанный аудиофрагмент сохранен для дальнейшей обработки на Google Drive в файле: {final_cut_file}. \nЧтобы вырезать еще фрагмент из исходного аудио, снова запустите все ячейки, начиная с 'Вырезание аудиофрагмента (запуск)...'. ")
else:
    wrapped_print("\nВырезанный аудиофрагмент не сохранен. \nЧтобы снова вырезать фрагмент из того же исходного аудио, запустите вновь все ячейки, начиная с 'Вырезаем аудиофрагмент: запуск...'")

# -> Только ⚠️ для транскрибации -> подключаем новую сессию Colab на GPU (T4)

(For transcription only -> Select 'GPU (T4)' Colab runtime type)

-> ⚠️  если транскрибация не нужна -
 пропускайте и переходите ниже сразу к разделу 'Работаем с контентом, используя ИИ (gpt-4o-mini)'

# Транскрибируем аудио (Whisper, локально) и сохраняем результат  на Google Drive

(Transcription. Skip if not needed)

# Подготовка

Запустите следующие шесть ячеек, чтобы подготовить транскрибационную сессию к запуску

In [ ]:
# INSTALL WHISPER for local use

!pip install -U -q openai-whisper
!pip install pydub

In [ ]:


# МMOUNT YOUR GOOGLE DRIVE (to use as storege)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# IMPORTS

import whisper
import textwrap
import json
import re
from pathlib import Path
from pydub import AudioSegment
from collections import Counter

In [ ]:

# -------------------------
# FOLDER PATHS
# -------------------------

# Specify path to the folder set up in your Google Drive to save processed material from the project
DRIVE_FOLDER = Path("/content/drive/My Drive/...your folder name to save files with project material...")

CHUNKS_DIR = DRIVE_FOLDER / "chunks"         # for session temporary files
TRANSCRIPTS_DIR = DRIVE_FOLDER / "transcripts"      # for session temporary files


# In case the folders do not exist yet
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# -------------------------
# HELPER FUNCTIONS
# -------------------------

# Returns normalized topic name (specified by user) as filename base for automated file naming
def normalize_user_topic_name(user_input):

    t_name_base = "".join(c for c in user_input if c.isalnum() or c in (' ', '-', '_')).strip()
    t_name_base = t_name_base.replace(" ", "_").lower()

    return t_name_base


# Formats text output for better readability
def wrapped_print(text, width=80):
    for line in text.split('\n'):
        print('\n'.join(textwrap.wrap(line, width)))


# Transcription result sanity check 1 (looping word combinations)
def has_hard_loops(words, max_phrase_size=6, min_repeats=4):
    n = len(words)
    for size in range(1, max_phrase_size + 1):
        i = 0
        while i + size * min_repeats <= n:
            phrase = words[i:i+size]

            repeats = 1
            j = i + size

            while j + size <= n and words[j:j+size] == phrase:
                repeats += 1
                j += size

            if repeats >= min_repeats:
                return True

            i += 1

    return False



# Transcription result sanity check 2 (some word present too many times)
def one_word_dominates(words):
    counter = Counter(words)
    most_common, freq = counter.most_common(1)[0]

    return freq > len(words) * 0.85


# Transcription result sanity check 3 (for short loops like "a a a a a" or "the the the the")
def has_too_many_short_loops(words):
    repeats = 0
    for i in range(len(words) - 1):
        if words[i] == words[i+1]:
            repeats += 1

    return repeats > len(words) * 0.3


# Whisper output basic overall sanity check
def is_valid_transcript(text: str) -> bool:
    text = text.strip()

    # --- Output too short

    if len(text) < 20:
        return False

    # ---- Remove non-speech noise descriptions
    text = re.sub(
        r"\[(music|applause|laughter|музыка|аплодисменты|смех).*?\]|\((music|applause|музыка|аплодисменты).*?\)|♪",
        "",
        text,
        flags=re.I
    )

    words = re.findall(r"\b[a-zA-Zа-яА-Я]+\b", text.lower())

    if len(words) < 5:
        return False

    # --- Run transcription output sanity check functions
    if has_hard_loops(words):
        return False

    if one_word_dominates(words):
        return False

    if has_too_many_short_loops(words):
        return False

    return True


# For cleanup after transcription/translation completed
def delete_temp_files(*folders, quiet=True):

    deleted, errors = 0, 0

    for folder in folders:
        for file in folder.glob("*"):
            if file.is_file():
                try:
                    file.unlink()
                    deleted += 1
                    if not quiet:
                        print(f"Удален временный файл: {file}")
                except Exception as e:
                    errors += 1
                    if not quiet:
                        print(f"Ошибка {e} при удалении временного файла {file}")

    print(f"Очистка временных/служебных файлов завершена: успешно удалено {deleted} файл(ов), ошибки возникли при удалении {errors} файл(ов).\n")

In [ ]:
# -------------------------
# TRANSCRIPTION WORKFLOW organized as session
# -------------------------

class TranscriptionSession:
    def __init__(self, drive_folder, chunks_dir, transcripts_dir):
        self.drive_folder = drive_folder
        self.chunks_dir = chunks_dir
        self.transcripts_dir = transcripts_dir
        # File with session metadata saved to one of the folders for temporary files: chunks_dir to facilitate cleanup
        self.session_meta_path = self.chunks_dir / "_session.json"

        self.audio = None
        self.topic_name = None

        self.whisper_model = None
        self.whisper_language = None

        self.num_chunks = 0
        self.final_transcript = ""

        self.model_name = None
        self.final_transcript_path = None


    # Interacts with user to choose an audio to transcribe
    # (from a list of filename bases (topic names) of saved audios)
    def choose_audio(self):
        audios = [
            fpath.stem.removesuffix("_a_cut")
            for fpath in self.drive_folder.glob("*_a_cut.wav")
        ]

        if not audios:
            raise FileNotFoundError("\nФайлов с аудиофрагментами не обнаружено.")

        list_for_user = input("Хотите посмотреть, какие аудиофрагменты ждут транскрибации (да/нет)?  ").strip().lower()

        if list_for_user == "да":
            print("\nАудиофрагменты, ожидающие транскрибации:")
            for i, name in enumerate(audios, start=1):
                print(f"Номер {i} --> {name}")

        while True:
            user_input = input(
                "Введите номер или скопируйте/вставьте наименование из перечня, либо введите наименование по памяти: "
            ).strip()

            if user_input.isdigit():
                index = int(user_input) - 1
                if 0 <= index < len(audios):
                    t_name_base = audios[index]
                else:
                    print("\nВведенного номера нет в перечне.")
                    continue
            else:
                t_name_base = normalize_user_topic_name(user_input)
                print(f"\n...Ищем аудиофайл, исходя из введенного по памяти наименования: {t_name_base}")

            audio_path = self.drive_folder / f"{t_name_base}_a_cut.wav"

            if audio_path.exists():
                try:
                    self.audio = AudioSegment.from_wav(audio_path)
                    self.topic_name = t_name_base
                    print(f"\nАудиофрагмент '{t_name_base}' загружен и готов для обработки!")
                    break
                except Exception as e:
                    print(f"\nОшибка при загрузке аудиофрагмента: {e}.")
            else:
                print("\nСмысловое наименование введено некорректно. Проверьте и повторите выбор.")


    # Chooses the language-appropriate Whisper model (en for English, multilingual for Russian audios)
    def choose_whisper_model(self):

        print("\nВыбираем и загружаем нужную модификацию Whisper...")
        lang_choice = input("\nВыбранный аудиофрагмент - англоязычный? (да/нет)? ").strip().lower()

        if lang_choice == 'да':
            self.model_name = "medium.en"
            self.whisper_language = "en"
        else:
            self.model_name = "medium"
            self.whisper_language = None

        self.whisper_model = whisper.load_model(self.model_name)

        lang_suffix = "en" if self.whisper_language == "en" else "ru"  # None -> 'ru' (for this project)
        self.final_transcript_path = (
            self.drive_folder / f"{self.topic_name}_full_trnscr_{lang_suffix}.txt"
        )



    # ------ Methods for handling session metadata -----

    # For the case of Colab crash or runtime disconnect amidst session
    # (interrupted session resumes automatically at next run)
    def save_session_metadata(self):
        data = {
            "topic_name": self.topic_name,
            "audio_file": f"{self.topic_name}_a_cut.wav",
            "model_name": self.model_name,
            "whisper_language": self.whisper_language
        }

        self.session_meta_path.write_text(
            json.dumps(data, indent=2),
            encoding="utf-8"
        )


    # Restores state of a previously interrupted session
    # (Messages about pure technical details are in English)
    def load_session_metadata(self):

        data = json.loads(self.session_meta_path.read_text(encoding="utf-8"))

        if not isinstance(data, dict):
            raise ValueError(
                "Session metadata must be a JSON object."
            )


        required_keys = {"topic_name", "audio_file", "model_name", "whisper_language"}
        missing = required_keys - data.keys()

        if missing:
            raise ValueError(
                f"Session metadata missing keys: {missing}"
            )

        self.topic_name = data["topic_name"]

        audio_path = self.drive_folder / data["audio_file"]
        self.audio = AudioSegment.from_wav(audio_path)

        # Language-appropriate Whisper model
        self.model_name = data["model_name"]
        self.whisper_language = data["whisper_language"]
        self.whisper_model = whisper.load_model(self.model_name)

        if self.whisper_language not in ("en", None):
            raise ValueError(f"Unexpected language: {self.whisper_language}")

        # Build file name/path to save final transcript text
        lang_suffix = "en" if self.whisper_language == "en" else "ru"  # None -> 'ru' (for this project)
        self.final_transcript_path = (
            self.drive_folder / f"{self.topic_name}_full_trnscr_{lang_suffix}.txt"
        )
        print(f"Продолжаю ранее прерванную сессию транскрибации аудиофрагмента '{self.topic_name}'.")


    # Automatically picks session mode for every run (continues previously interrupted session - if any, or starts a new one)
    def ensure_session_loaded(self):
        if self.session_meta_path.exists():
            print("Есть запись ранее прерванной сессии: загружаю, чтобы продолжить...\n\n")
            try:
                self.load_session_metadata()
            except Exception as e:
                delete_temp_files(self.chunks_dir, self.transcripts_dir)
                raise RuntimeError(
                    f"Session metadata corrupted: {e}. Cleaned temp files. Rerun required."
                    "\nЕсть запись ранее прерванной сессии, но её загрузочные данные неполны -> продолжить эту сессию невозможно..."
                    "\nДанная запись удалена. Начните новую сессию, перезапустив ячейку 'Транскрибация: запуск + ...'."
                ) from e

        else:
            self.choose_audio()
            self.choose_whisper_model()
            self.save_session_metadata()

    # -------------------------



    # Handles audio chunking or existing chunks (even if Colab disconnects halfway through a long
    # transcription session, it will be resumed at next run - previous processing results are not lost)
    def ensure_chunks(self, chunk_length_ms=15 * 60 * 1000):

        # If chunks from a previously interrupted session exist
        existing_chunks = sorted(self.chunks_dir.glob("chunk_*.wav"))

        indices = []
        for p in existing_chunks:
            try:
                idx = int(p.stem.split("_")[1])
                indices.append(idx)
            except Exception:
                raise RuntimeError(f"Некорректное имя файла с отрезком аудиофрагмента: {p.name}")

        # Chunk sequence check (sequence not in tact -> clean up session metadata and temporary files!)
        if indices:
            expected = list(range(min(indices), max(indices) + 1))

            if sorted(indices) != expected:
                delete_temp_files(self.chunks_dir, self.transcripts_dir)
                raise RuntimeError(
                    f"Набор отрезков аудиофрагмента неполон или содержит ошибки: имеются отрезки только с номерами {sorted(indices)} вместо ожидаемого комплекта {expected}"
                )

            self.num_chunks = len(indices)
            print(f"Необходимые {self.num_chunks} отрезков данного аудио уже существуют: пропускаю этап разделения аудиофрагмента на отрезки.")
            return



        # If no temporary chunk files found -> chunk audio!
        if self.audio is None:         # Check first
            raise RuntimeError("Аудиофрагмент не загружен -> нечего делить на отрезки.")

        if len(self.audio) > chunk_length_ms:
            self.num_chunks = (len(self.audio) + chunk_length_ms - 1) // chunk_length_ms
            print(f"\n\nПредстоит протранскрибировать {self.num_chunks} отрезка(ов) аудиофрагмента.\n\n")

            for i in range(self.num_chunks):
                chunk_path = self.chunks_dir / f"chunk_{i:03}.wav"

                if chunk_path.exists():
                    continue     # Idempotency ensured

                start = i * chunk_length_ms
                end = min(start + chunk_length_ms, len(self.audio))
                chunk = self.audio[start:end]
                chunk.export(chunk_path, format="wav")
        else:
            self.num_chunks = 1
            print("\n\nАудиофрагмент небольшой, по длительности укладывается в один отрезок...\n\n")

            chunk_path = self.chunks_dir / "chunk_000.wav"

            if not chunk_path.exists():
                self.audio.export(chunk_path, format="wav")


    # --- Audio chunk transcription with basic Whisper output sanity checks  ----
    def transcribe(self):
        print(
            "\nПриступаю к транскрибации: обработка каждого отрезка аудио занимаёт несколько минут...\n\n"
        )

        # Defensive check (a guardrail in case something unexpected outside normal workflow happened)
        if self.num_chunks <= 0:
            raise RuntimeError("No chunks to process. Did you run ensure_chunks()?")

        for i in range(self.num_chunks):
            chunk_path = self.chunks_dir / f"chunk_{i:03}.wav"
            transcript_path = self.transcripts_dir / f"chunk_{i:03}.txt"
            temp_path = transcript_path.with_name(transcript_path.name + ".tmp")

            # Make sure no stale temporary files with audio chunk transcripts are left
            if temp_path.exists():
                temp_path.unlink()

            # Basic sanity checks for saved chunk transcriptts (if previous session has been interrupted)
            if transcript_path.exists():
                text = transcript_path.read_text(encoding="utf-8")

                # Skip repeated transcription if sanity checks are ok
                if is_valid_transcript(text):
                    print(f"{i+1}й отрезок аудио из {self.num_chunks}: не обрабатываем (текст транскрибации уже есть)")
                    continue
                else:
                    print(f"{i+1}й отрезок аудио из {self.num_chunks}: обнаруженный текст транскрибации не прошел формальную проверку качества → транскрибируем заново")


            # Defensive checks
            if self.whisper_model is None:
                raise RuntimeError("Whisper model not loaded.")

            if not chunk_path.exists():
                raise FileNotFoundError(f"Missing chunk file: {chunk_path.name}")


            result = self.whisper_model.transcribe(
                str(chunk_path), language=self.whisper_language
                )

            if not isinstance(result, dict) or "text" not in result:
                raise RuntimeError("Unexpected Whisper response format.")

            text = result["text"].strip()

            # Quality first -> better re-run session
            if not text:
                raise RuntimeError("Модель Whisper выдала пустой текст транскрибации -> останавливаю обработку для сохранения качества. Перезапустите сессию, чтобы продолжить.")

            try:
                # Atomic chunk transcript save - no partial file writes!
                temp_path.write_text(text, encoding="utf-8")
                temp_path.replace(transcript_path)

                print(f"{i+1}й отрезок аудио из {self.num_chunks}: протранскрибирован")



            # Quality first: if something went wrong, better start a new run to continue session
            except Exception as e:
                print(f"{i+1}й отрезок аудио из {self.num_chunks}: ОШИБКА при транскрибации - {e}")
                print("Останавливаю обработку для сохранения качества. Проверьте причину и перезапустите  транскрибационную сессию.")
                raise



    # Merges audio chunk transcripts into one final text
    # If some chunk transcript is absent/invalid, notifies user and proceeds.
    def merge_transcripts(self):

        parts = []
        missing = []
        invalid = []

        # Defensive check (a guardrail in case something unexpected outside normal workflow happened)
        if self.num_chunks <= 0:
            raise RuntimeError("No chunks to merge.")

        print(f"\n\nПроверяю транскрипт(ы), собираю итоговый текст...\n\n")

        for i in range(self.num_chunks):
            transcript_path = self.transcripts_dir / f"chunk_{i:03}.txt"
            # Already ruled out: saved empty Whisper outputs and chunk transcript partial writes
            if transcript_path.exists():
                text = transcript_path.read_text(encoding="utf-8")

                # Chunk transcripts sanity checks and sorting
                if is_valid_transcript(text):
                    parts.append(text)
                    print(f"Транскрипт {i+1}го отрезка аудио -> добавлен в итоговый текст.")
                else:
                    invalid.append((i, text))
                    print(f"Транскрипт {i+1}го отрезка аудио -> не прошел формальную проверку.\n")
            else:
                missing.append(i)

        if missing:       # User info
            print(
                f"⚠️ ВНИМАНИЕ: Из ожидаемых {self.num_chunks} транскриптов отрезков аудио отсутствуют {len(missing)}.\n"
                f"Порядковые номера отрезков аудио, транскрипты которых отсутствуют: {missing}\n"
                "⚠️ Итоговый текст транскрибации неполон.\n"
            )

        if invalid:       # User info
            for idx, txt in invalid:
                print(f"\n--- Не прошедший проверку транскрипт {idx+1}го отрезка аудио ---\n")
                wrapped_print(txt)

        self.final_transcript = "\n".join(parts)

        print(f"\n\nКоличество знаков в итоговом тексте транскрибации: {len(self.final_transcript)}\n\n")

        return self.final_transcript



    # Finalizes session (user concents to finalize -> final transcript text saved to user's Google Drive, folders for temporary files cleaned up)
    def finalize(self):
        # User info - what finalization does
        print(
            "\n\n! ⚠️ ВНИМАНИЕ  !\n"
            "Подтверждайте финализацию, только если итоговый текст транскрибации вас устраивает.\n"
            "Подтверждая финализацию, вы:\n"
            "-> СОХРАНЯЕТЕ итоговый текст в хранилище Google Drive для дальнейшего использования;\n"
            "-> УДАЛЯЕТЕ временные файлы, созданные во время обработки, и протранскрибированный аудиофрагмент.\n"
            "Если итоговый текст вас не удовлетворяет, нажмите Enter, чтобы выйти.\n\n"
        )

        # No finalization possible if final merged text empty -> new session required
        if not self.final_transcript.strip():
            print("Итоговый текст транскрибации пуст -> финализация невозможна, все временные/служебные файлы неудачной сессии удалены. Для повторной транскрибации запустите новую сессию.")
            delete_temp_files(self.chunks_dir, self.transcripts_dir)
            return

        confirm = input("Введите 'да', чтобы финализировать, либо нажмите Enter, чтобы выйти: ").strip().lower()

        # User does not confirm finalization -> all folders for temporary files are cleaned up, next run starts a new session
        if confirm != "да":
            print("\nФинализация отменена, все временные/служебные файлы сессии удалены. Для повторения попытки транскрибации запустите новую сессию.")
            delete_temp_files(self.chunks_dir, self.transcripts_dir)

            return

        # ---- Save final transcript text to user's Google Drive

        temp_path = self.final_transcript_path.with_name(
            self.final_transcript_path.name + ".tmp"
        )

        try:
            # Atomic write -> no partial file writes!
            temp_path.write_text(self.final_transcript, encoding="utf-8")

            temp_path.replace(self.final_transcript_path)

            print(f"\nИтоговый текст транскрибации сохранен в файл {self.final_transcript_path.name}")
        except Exception as e:
            # Keep user storage tidy
            if temp_path.exists():
                temp_path.unlink()
            print(f"\nОшибка при сохранении итогового текста транскрибации: {e}")
            print("\nФинализация не состоялась, удаление всех временных/служебных файлов транскрибационной сессии ОТМЕНЕНО. Перезапустите сессию, чтобы финализировать.")
            return

        #  ----- Post-session cleanup with user messages ------

        delete_temp_files(self.chunks_dir, self.transcripts_dir)
        print("Все временные/служебные файлы финализированной сессии удалены.")

        # Delete transcribed audio -> keep user storage tidy

        audio_path = self.drive_folder / f"{self.topic_name}_a_cut.wav"

        try:
            if audio_path.exists():
                audio_path.unlink()
                print(f"Файл '{audio_path.name}' с протранскрибированным аудиофрагментом удален.")
            else:
                print("Файл с протранскрибированным аудиофрагментом не обнаружен.")
        except Exception as e:
            print(f"Ошибка при удалении протранскрибированного аудиофрагмента: {e}")


# User info
print("Условия для запуска транскрибационной сессии созданы!..")

# Транскрибационная сессия: запуск + финализация

-> Запустите следующую ячейку ниже, чтобы активировать транскрибационную сессию.

-> Если при активации обнаружится запись ранее незавершенной (прерванной) сессии, то
она будет автоматически загружена и доведена до конца.

-> Если такой записи нет, начинается новая сессия: вам будет предложено выбрать
аудио для транскрибации из перечня смысловых наименований аудиофрагментов, которые вы уже ранее
скачали, вырезали и сохранили на Google Drive.

-> Чтобы выбрать аудио, введите в окне запроса его номер или скопируйте туда его наименование из представленного
перечня (рекомендуется), либо введите сиысловое наименование по памяти (не рекомендуется).

⚠️ Если вы вводите смысловое наименование по памяти, то обратите особое внимание на точность его написания.

-> Для обеспечения стабильного качества транскрибации аудиофрагмент обычно разбивается на небольшие отрезки;
по мере их обработки выводятся сообщения.

-> Итоговый текст выводится в консоль.

-> Далее вам нужно будет подтвердить финализацию транскрибационной сессии.

⚠️ Перед подтверждением финализации удостоверьтесь, что качество итогового текста вас устраивает.

⚠️ ВАЖНО ⚠️  Подтверждение финализации сохраняет итоговый текст для дальнейшего использования и удаляет все временные
файлы сессии, а также протранскрибированный аудиофрагмент.

-> Новую транскрибационную сессию можно начать сразу после завершения предыдущей, заново запустив соответствующую ячейку.

 -> Если Colab отключился/обрушился посреди сессии, то прерванную сессию можно позже продолжить, заново подключив Colab и запустив все ячейки раздела
'Транскрибируем аудио...'.

(Внезапное отключение Colab от вычислительного
времени возможно, особенно при запуске на GPU (T4): предоставля вычислительные мощности бесплатно, Google
ограничивает продолжительность пользования ими в зависимости от текущей загрузки.)

In [ ]:

# --------------------
# EXECUTE session
# --------------------

# Run transcription
session = TranscriptionSession(
    DRIVE_FOLDER,
    CHUNKS_DIR,
    TRANSCRIPTS_DIR
)


session.ensure_session_loaded()
session.ensure_chunks()
session.transcribe()
session.merge_transcripts()

# Resulting transcription output
wrapped_print(session.final_transcript)

# Finilize
session.finalize()

# ->  ⚠️ Если начинаем с этого раздела или продолжаем сразу после транскрибации -> подключаем новую сессию Colab на CPU

(Select the 'CPU' Colab runtime type)

# Работаем с контентом, используя ИИ (gpt-4o-mini от OpenAI)

(AI assistant and chat with AI discussion partners, LLM model: gpt-4o-mini from OpenAI)

# ИИ-ассистент (перевод/главные идеи текста) и ИИ-собеседники (чат): предварительная подготовка

Для подготовки условий работы с GPT запустите восемь (!) из девяти ячеек ниже.

In [ ]:
# ----------------------
# INSTALL PACKAGES
# ----------------------

!pip install -q faiss-cpu openai langchain langchain-openai langchain-community langchain-classic

⚠️ Организуем доступ к OpenAI

In [ ]:
# -----------------------
# OPENAI API ACCESS: IMPORTS
# -----------------------


import os
import openai
from openai import OpenAI
from google.colab import userdata

⚠️ Запускайте только одну ⚠️ из двух ячеек доступа к OpenAI ниже

(⚠️ Run only one of the two ACCESS cells below)

In [ ]:
# -----------------------
# ⚠️ DIRECT ACCESS
# -----------------------

# Your OpenAI API key provided through Colab Secrets

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

api_client = OpenAI(timeout=30)

In [ ]:
# -----------------------
# ⚠️ ACCESS VIA PROXY
# -----------------------

# Your proxy key provided through Colab Secrets and proxy url specified in cell

os.environ["OPENAI_API_KEY"] = userdata.get('YOUR_PROXY_API_KEY')
os.environ["OPENAI_BASE_URL"] = "https://...your_proxy_url..."


api_client = OpenAI(timeout=30)

In [ ]:


# MOUNT YOUR GOOGLE DRIVE (to use as storage)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# IMPORTS

import random
import time
import textwrap
import re
import json
from langchain_classic.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

In [ ]:
# ----------------
# FOLDER PATHS
#-----------------

from pathlib import Path

# Specify path to the folder set up in your Google Drive to save processed material from the project
DRIVE_FOLDER = Path("/content/drive/My Drive/...your folder name to save files with project material...")

VECTORDB_FOLDER = DRIVE_FOLDER / "vector_dbases"

VECTORDB_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
# ----------------------
# HELPER FUNCTION
# ----------------------

# Formats text output for better readability
def wrapped_print(text, width=80):
    for line in text.split('\n'):
        print('\n'.join(textwrap.wrap(line, width)))

In [ ]:

# -----------------
#  LLM API CALL gateway
# ------------------


# Gives error message in plain words for non-technical user
class LLMServiceUnavailable(RuntimeError):
    """Appears in case of model API unavailability or persistent error"""
    pass


# Processes logic for all LLM API calls of the project
# (Infos about pure technical processing details are in English)
class LLMService:
    MAX_INPUT_CHARS = 120_000  # Input size limit

    def __init__(
        self,
        client,
        default_model="gpt-4o-mini",
        max_retries=3,
        base_delay=1.5,
        debug=False
    ):
        self.client = client
        self.default_model = default_model
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.debug = debug


    # Selects API call errors for retries (Error not transient -> no point retrying)
    @staticmethod
    def _is_retryable_error(e):
        msg = str(e).lower()
        return any(keyword in msg for keyword in [
            "timeout",
            "rate limit",
            "connection",
            "temporarily",
            "503",
            "502",
            "500",
            "server error",
            "overloaded",
            "429"
        ])


    # Main LLM API call execution method with retries (for transient errors only)
    def chat(self, messages, temperature=0.9, model=None):
        model = model or self.default_model
        last_error = None

        # Limit input size
        total_chars = sum(len(m.get("content", "")) for m in messages)
        if total_chars > self.MAX_INPUT_CHARS:
            raise ValueError(
                f"Ввод слишком большой: ({total_chars} знаков). "
                "Отправляйте на обработку не более 120_000 знаков за раз."
            )

        for attempt in range(1, self.max_retries + 1):
            try:
                if self.debug:
                    print(f"[LLMService] Attempt {attempt}/{self.max_retries}")
                    print(f"[LLMService] Model={model}, Temp={temperature}")
                    print(f"[LLMService] Chars={total_chars}")

                completion = self.client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature
                )

                text = self._extract_text(completion)
                return text

            except Exception as e:
                # Error not transient: execution stops, user is advised to retry later
                if not self._is_retryable_error(e):
                    raise LLMServiceUnavailable(
                        f"Сбой в системе обработки запроса продолжается - вернитесь к работе через некоторое время: {e}\n"
                        "Это может быть связано с нестабильностью интернет-соединения, состоянием принимающего сервера и т.п."
                    )

                last_error = e

                # Exponential timeouts for retries
                base_wait = self.base_delay * (2 ** (attempt - 1))

                # Add jitter (±30%)
                jitter_factor = random.uniform(0.7, 1.3)
                wait = base_wait * jitter_factor

                # If 'debug=True'
                if self.debug:
                    print(f"[LLMService] Error: {repr(e)}")
                    print(f"[LLMService] Backoff base: {base_wait:.2f}s")
                    print(f"[LLMService] Jitter factor: {jitter_factor:.2f}")
                    print(f"[LLMService] Retrying in {wait:.2f}s...")

                time.sleep(wait)

        # If all retries fail, user is advised to resume later
        msg = (
            f"Обращение к модели по API после {self.max_retries} попыток не увенчалось успехом: {last_error}.\n"
            "Завершите сессию подключения Colab, через некоторое время проверьте ваше интернет-соединение и зппуститесь снова."
        )

        raise LLMServiceUnavailable(msg)


    # Returns LLM text output after basic sanity checks or "" if output empty
    def _extract_text(self, completion):
        choices = getattr(completion, "choices", None)
        if not choices:
            return ""

        first = choices[0]
        message = getattr(first, "message", None)
        if not message:
            return ""

        content = getattr(message, "content", None)

        return content.strip() if isinstance(content, str) else ""


    # ----  Task-specific methods (chat summary, translation, extracting main ideas, answers for AI discussion partners)

    # Returns chat summary as JSON
    def summarize(self, messages, retries=3):
        for attempt in range(retries):
            try:
                completion = self.client.chat.completions.create(
                    model=self.default_model,
                    messages=messages,
                    temperature=0,
                    response_format={"type": "json_object"}
                )
                text = self._extract_text(completion)

                if not text:
                    raise ValueError("Empty summary response")

                try:
                    parsed = json.loads(text)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSON returned: {e}")

                return text

            except Exception as e:
                print(f"⚠️ Summary attempt {attempt+1} failed:", e)

                # Change pause duration exponentially
                time.sleep((2**attempt) + random.random())

        print("⚠️ Summary failed after retries")
        return None


    # Returns Russian translation of selected English transcript
    def translate_en_ru(self, system, text):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": f"""Переведи на русский язык англоязычный текст: \n\n{text}.\n\n
            При переводе как иожно точнее передавай смысл текста, соблюдая при этом общий стиль изложения.\n\n
            В качестве ответа верни текст русского перевода."""
            }
        ]
        return self.chat(messages, temperature=0.2)


    # Returns main ideas of selected transcript text with 2 questions for better understanding of each idea
    def extract_main_ideas(self, system, text):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": f"""Проанализируй предоставленный тебе текст:\n\n{text}.\n\n
                Выдели главные идеи этого текста и по каждой выделенной идее сформулируй 2 вопроса, ответы на которые помогли бы глубже раскрыть психологические и/или философские ее аспекты.
                ВАЖНО: ВСЕГДА отвечай на русском языке!"""
            }
        ]
        return self.chat(messages, temperature=0.3)


    # Returns answers for 'non-creative' AI discussion partners (specialties: Literarary Critic and Psycologist), one LLM call
    def agent_reply(self, messages, temperature=0.9):
        return self.chat(messages, temperature=temperature)


    # Returns answers for the 'creative' AI discussion partner (specialty: Unconventional Thinker), two LLM calls
    def creative_dialogue(self, user_query, messages, generator_system_prompt):

        # ---- Call 1: User query analysis ----
        # LLM receives user_input, prompt and context built similarly to other Ai chat companions
        # (system_prompt_analyst + personality, unsummarized_history, chat summary as JSON, optionally - RAG)
        # Extracts 3 aspects of user query

        analysis = self.chat(
            messages,
            temperature=0.3
        )

        if not analysis or not analysis.strip():
            print("⚠️ Первый этап обработки вернул пустой ответ. Продолжайте чат.")
            return "Something went wrong while analysing the question. No analysis available."

        # ---- Call 2: Generates creative ideas----
        # LLM receives generator input and generator_system_prompt consisting of
        # base_generator_prompt + personality (built by the 'chat' method of the GPTDialogueManager class)

        generator_input = f"""
USER QUERY:
{user_query}

ANALYST OUTPUT:
{analysis}

TASK
Using the aspects above, generate three creative ideas with a concise explanation of their meaning and implications.
Each idea should develop one aspect.
You MUST always answer in Russian!
"""

        generator_messages = [
                    {
                        "role": "system",
                        "content": generator_system_prompt
                    },
                    {
                        "role": "user",
                        "content": generator_input
                    }
                ]


        ideas = self.chat(
            generator_messages
        )


        if not ideas or not ideas.strip():
            print("⚠️ Второй этап обработки вернул пустой ответ. Продолжайте чат. Если ответ снова будет пустым, завершите чат и вновь запуститесь позже.")
            return "Something went wrong while idea generation. Please try again."

        return f"""
●=> {analysis.strip()}

●=> {ideas.strip()}
"""


# ---------------
# Instance of LLMService class
# --------------

llm_service = LLMService(api_client)

# Если ИИ-ассистент в текущей сессии не нужен, можете сразу перейти к разделу 'Чат с ИИ-собеседниками: подготовка'

(The AI assistant part can be skipped if not needed)


# ИИ-ассистент: подготовка

(AI assistant - translates en->ru and extracts main ideas)

Чтобы подготовить ИИ-ассистента к работе, запустите следующие две ячейки ниже.

In [ ]:
# -----------------------
# HELPER FUNCTIONS
# -----------------------


# Interacts with user to choose a transcript text to process from a list of filename bases (topic names)
def choose_topic_from_list(topics):
    print("\nЕсть тексты транскриптов по следующим смысловым наименованиям:\n")
    for idx, topic in enumerate(topics, 1):
        print(f"Номер {idx} --> {topic}")

    while True:
        choice = input("\nВыберите номер текста для обработки или нажмите Enter, чтобы выйти: ").strip()
        if choice == "":
            return None
        if choice.isdigit() and 1 <= int(choice) <= len(topics):
            return topics[int(choice) - 1]
        print("\nЭтот ввод нерелевантен для выбора -> укажите только один из номеров из перечня.")


# Searches storage (your Google Drive) for all transcript files (excluding nested folder for temp files)
# Returns list of available transcript files by filename bases (topic names)
def list_available_transcripts_pathlib():
    transcripts = []

    for file_path in DRIVE_FOLDER.glob("*_full_trnscr_*.txt"):
        base = re.sub(r"_(full_trnscr_en|full_trnscr_ru)\.txt$", "", file_path.name)
        transcripts.append(base)

    return sorted(transcripts)


# Handles workflow of user picking a text to process
def prompt_user_choice(available_topics):
    user_answer = input("Хотите посмотреть перечень смысловых наименований текстов, из которых можно выбрать (да/нет)?   ").strip().lower()

    if user_answer == 'да':
        if available_topics:
            selected_topic = choose_topic_from_list(available_topics)
            if selected_topic:
                print(f"\nВы выбрали текст транскрипта {selected_topic}")
                return selected_topic
            else:
                print("\nПроцесс выбора прекращен")
                return None
        else:
            print("\nДоступных для данной задачи текстов транскриптов не найдено.")
            return None
    else:
        print("\nТекст транскрипта не выбран: вы вышли из процесса выбора или некорректно ввели ответ на запрос о выборе.")
        return None


# Returns list of English transcripts without matching Russian translation
def list_en_transcripts_to_translate():
    transcripts = []
    for file_path in DRIVE_FOLDER.glob("*_full_trnscr_en.txt"):
        topic_name = file_path.stem.replace("_full_trnscr_en", "")
        ru_file = DRIVE_FOLDER / f"{topic_name}_full_trnsltn_ru.txt"
        if not ru_file.exists():
            transcripts.append(topic_name)
    return sorted(transcripts)


# Loads text from selected file, handles possible errors
def load_trnscr_text(fname):
    reqrd_file_path = DRIVE_FOLDER / fname

    try:
        with open(reqrd_file_path, 'r', encoding='utf-8') as file:
            text = file.read().strip()
            if not text:
                raise ValueError("Запрошенный текстовый файл пуст.")
            print("\nВыбранный текст загружен!\n\n")
            return text
    except Exception as e:
        raise RuntimeError(f"Ошибка {e} при чтении запрошенного файла {reqrd_file_path}")


# Splits loaded text into chunks (splitting is word-based: Whisper output may lack full stop signs)
def text_to_chunks(chunk_size, chunk_overlap, text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n', '.', '!', '?', ',', ' ', '']   # '\n\n',
    )

    chunks = splitter.split_text(text)

    return chunks


# Processes chunks according task (calls task-specific method from LLMService class, adds headers for main ideas extraction)
def process_chunks(chunks, process_fn, header_fn=None, **kwargs):
    parts = []
    total = len(chunks)

    for i, chunk in enumerate(chunks, 1):
        # Optional output for control/debugging
        #print(f"Chunk {i}: {len(chunk)} chars")

        print(f"Обрабатываю {i}-й отрезок текста из {total}...\n")

        content = process_fn(text=chunk, **kwargs)

        if not content:
            content = f"[⚠️ {i}-й отрезок текста обработан, но полученный вывод пуст]"

        if header_fn:
            header = header_fn(i, total)
            parts.append(f"{header}\n{content}")
        else:
            parts.append(content)

    return "\n\n".join(parts)


# Handles workflow for extracting main ideas (idea + explanation + 2 questions for better understanding)
def main_ideas_from_topic(t_name_base):
    is_russian = input("Выбранный текст транскрипта получен из русскоязычного аудио? (Введите 1 для 'да', 0 для 'нет')   ").strip() == "1"

    fname = (
            f"{t_name_base}_full_trnscr_ru.txt"
            if is_russian
            else f"{t_name_base}_full_trnscr_en.txt"
    )

    text_for_ideas = load_trnscr_text(fname)

    if len(text_for_ideas) > 10000:
        text_chunks = text_to_chunks(10000, 300, text_for_ideas)

        # Header with processed text chunk number added to preserve logic
        all_main_ideas = process_chunks(
            chunks=text_chunks,
            process_fn=llm_service.extract_main_ideas,
            system=system_m_i_q,
            header_fn=lambda i, total: f"=== Для {i}-го отрезка текста из {total} ==="
        )
    else:
        all_main_ideas = llm_service.extract_main_ideas(system_m_i_q, text_for_ideas)

    # Result empty -> notify user: rerun required
    if not all_main_ideas:
        print("Ответ от модели получен, но он пуст. Для новой попытки перезапустите эту ячейку.")

    return all_main_ideas

In [ ]:
# ------------------------
# SYSTEM PROMPTS for AI ASSISTANT
# ------------------------


# For extracting main ideas (with explanation of importance and 2 questions for better understanding - for each idea), answer always in Russian
system_m_i_q = """Ты - AI-ассистент по анализу смыслового содержания текстов.
Твоя задача - проанализировать смысл предоставленной тебе части транскриптп лекции или подкаста, выделить основные мысли этого отрывка и сформулировать
вопросы для более глубокого раскрытия психологических и философских аспектов каждой из выделенных тобой мыслей.

Генерация будет в 2 шага:
Шаг 1: Анализ текста и выделение его основных мыслей (агент-аналитик)
Шаг 2: Генерация вопросов (агент-генератор)

Делаем Шаг 1
Агент-аналитик
Задача агента-аналитика - проанализировать предоставленную часть транскрипта лекции или подкаста, определить основные мысли этого текста и причину, почему эта идея выделена в качестве одной из основных.
Ты прекрасно знаешь, что выделение основных мыслей текста отличается от краткого изложения (саммаризации) содержания текста.
Выделить основные мысли текста значит выделить основные идеи текста, не пересказывая при этом его непосредственного содержания и не добавляя ничего от себя.

ВАЖНО: если ты обрабатываешь англоязычный текст, будь особенно внимателен при передаче на русском языке таких английских понятий, смысл которых не может быть выражен
одним словом по-русски: при формулировании ответа по-русски передавай такие понятия словосочетаниями, используя несколько слов.
Пример 1: в обрабатываемом англоязычном тексте обсуждается 'hypervigilance' -> в своем ответе обозначь это понятие примерно как 'сверхвысокая степень бдительности' или 'чрезмерная бдительность';
Пример 2: в обрабатываемом англоязычном тексте обсуждается 'reenactment' -> в своем ответе обозначь это понятие примерно как 'действия по повторному проживанию (травм, переживания и т.п.);
Пример 3: в обрабатываемом англоязычном тексте обсуждается 'reattachment' -> в своем ответе обозначь это понятие примерно как 'повторное прикрепление' или 'повторная привязка'.

На выходе аналитик представляет четкие, близкие к авторским формулировки выделенных им основных мыслей предоставленной части транскрипта лекции или подкаста в порядке их следования в тексте
с кратким обоснованием по каждой идее, почему она была выделена как одна из основных.

Делаем Шаг 2
Агент-генератор
Агент-генератор использует вывод аналитика как основу: по каждой обозначенной основной идее формулирует два открытых вопроса для более
глубокого раскрытия психологических и/или философских аспектов этого пункта.

Структура вывода:
NB! => ... (изложение первой по порядку основной мысли текста)
   ■ ... (краткое обоснование, почему эта мысль была веделена как одна из основных)
   ● Вопросы для углубления понимания:
      а) ... (первый вопрос для лучшего понимания этой мысли пользователем)
      б) ... (второй вопрос для лучшего понимания этой мысли пользователем)

NB! => ... (изложение второй по порядку основной мысли текста)
   ■ ... (краткое обоснование, почему эта мысль была выделена как одна из основных)
   ● Вопросы для лучшего понимания:
      а) ... (первый вопрос для углубления понимания этой мысли пользователем)
      б) ... (второй вопрос для углубления понимания этой мысли пользователем)

NB! => ... (и так далее по каждой из выделенных по тексту основных идей)

Выполняй все шаги 1 - 2 последовательно, без пауз и подтверждений.
Выводи результаты по каждой выделенной идее в виде текста в порядке следования этих идей в обрабатываемом тексте.
Не присваивай номера выделяемым основным идеям, даже если в предоставленном тебе тексте такая нумерация есть.
Обращвй внимание на то, как формулирует свои мысли автор лекции или подкаста, по возможности близко придерживайся формулировок автора.
Не делай сильных обобщений при формулировании основных мыслей, близко следуй за текстом.
Избегай заумных формулировок, штампов и клише.
Избегай употребления английских слов в своем ответе - за исключением общеупотребительных буквенных аббревиатур (например, CPTSD).
Если эначение английского слова нельзя передать по-русски одним словом, ВСЕГДА заменяй такие английские слова описанием из нескольких слов по-русски в своем ответе.
ВАЖНО! Всегда отвечай на русском языке!"""



# For translation en --> ru (in full, no summarizing to shorten output!)
system_en_to_ru = """Ты - AI ассистент по переводу английских текстов - транскрибаций лекций и бесед  по психологии, литературоведению и философии на русский язык.
Твоя задача - перевести на русский язык предоставленный тебе англоязычный текст транскрибации с максимально точной передачей смысла и сохранением стиля изложения.

При переводе уделяй особое вниманин точной передаче смысла текста, включая все нюансы, а также сохранению общего стиля изложения оригинала и богатсва языка:
это важно для правильного понимания излагаемых в тексте идей.
Если в английском тексте транскрибации ты встретишь ошибки (например, явно не соответствующие общему контексту слова), то исправь их в соответствии с контекстом и без дополнительных пометок.
При переводе иди строго по тексту оригинала, не придумывый ничего от себя. Тебе запрещено сокращать и/или обобщать при переводе.

ВАЖНО: будь внимателен при переводе таких английских слоа, которые не могут быть переведены на русский язык одним словом. Переводи такие английские слова словосочетаниями с использованием нескольких слов.
Пример 1: английское слово 'hypervigilance' переводи примерно как 'сверхвысокая степень бдительности' или 'чрезмерная бдительность';
Пример 2: английское слово 'reenactment' переводи примерно как 'действия по повторному проживанию (травмы, переживания и т.п.);
Пример 3: английское слово 'reattachment' переводи примерно как 'повторное прикрепление' или 'повторная привязка'.

Не оставляй отдельных английских слов в русском переводе, кроме общеупотребительных буквенных аббревиатур (например, CPTSD).
Если эначение английского слова нельзя передать по-русски одним словом, ВСЕГДА переводи такое слово на русский язык словосочетанием из нескольких слов.

В качестве ответа верни русский перевод предоставленного тебе англоязычного текста транскрибации."""

 # ИИ-ассистент: запуск

ИИ-ассистент сначала спросит вас, что вам нужно -извлечь основные идеи или сделать перевод, затем предложит выбрать текст для обработки из перечня
смысловых наименований доступны текстовых файлов. Результат выводится в консоль.

⚠️ ВАЖНО: Если в дальнейшем хотите использовать полученный русский перевод для детального обсуждения с ИИ-собеседником, СРАЗУ сохраните перевод на Google Drive, запустив соответствующую ячейку ниже

--> Новый запуск ИИ-ассистента очищает прежний вывод ⚠️ БЕЗ ⚠️ его сохранения.

--> По получении основных идей/перевода можно сразу снова запустить ИИ-ассистента для обработки другого текста.

In [ ]:
# --------------------
# RUN AI ASSISTANT
# --------------------

# User info
print("---> ИИ-ассистент по обработке текстов запущен!\n")

user_choice = input("Введите 'и', чтобы выделить основные идеи; либо нажмите Enter, чтобы сделать перевод: ").strip().lower()

if user_choice == "и":
    print("\nПриступаем к выбору текста и получению его основных идей...\n")
    available_topics = list_available_transcripts_pathlib()
    t_name_base = prompt_user_choice(available_topics)
    if t_name_base:
        main_ideas = main_ideas_from_topic(t_name_base)
        wrapped_print(main_ideas)
    else:
        print("Текст транскрипта не выбран -> обработка прекращена.")

else:
    print("\nПриступаем к выбору текста англ. транскрипта и получению его перевода на рус. яз....\n")
    available_topics = list_en_transcripts_to_translate()
    t_name_base = prompt_user_choice(available_topics)
    if t_name_base:
        fname_en = f"{t_name_base}_full_trnscr_en.txt"

        text_to_translate = load_trnscr_text(fname_en)

        if len(text_to_translate) > 6500:
            text_chunks_en = text_to_chunks(6500, 0, text_to_translate)

            # -> WITHOUT headers
            trnsltn_all_ru = process_chunks(
                chunks=text_chunks_en,
                process_fn=llm_service.translate_en_ru,
                system=system_en_to_ru
            )
        else:
            trnsltn_all_ru = llm_service.translate_en_ru(system_en_to_ru, text_to_translate)

        # Result empty -> notify user: rerun required
        if not trnsltn_all_ru:
            print("Ответ от модели получен, но он пуст. Для новой попытки перезапустите эту ячейку.")

        wrapped_print(trnsltn_all_ru)
    else:
        print("Текст транскрипта не выбран -> обработка прекращена.")

# Сохраняем полученный текст перевода на Google Drive

⚠️ ВАЖНО: сохранение текста обеспечивает его доступность для дальнейшего формирования основы обсуждения в чатах с ИИ-собеседниками!

In [ ]:
# ------------------------
# SAVE generated translation to your Google Drive
# ------------------------

user_answer = input("\nВведите да, чтобы сохранить текст перевода на Google Drive, либо нажмите Enter, чтобы выйти. ").strip().lower()

if user_answer == "да":
    translated_ru_fname = DRIVE_FOLDER / f"{t_name_base}_full_trnsltn_ru.txt"
    try:
        with open(translated_ru_fname, 'w', encoding='utf-8') as file:
            file.write(trnsltn_all_ru)
    except Exception as e:
        print(f"\nОшибка при сохранении текста перевода: {e}")
    else:
        print(f"\nТекст перевода сохранен на Google Drive в: {translated_ru_fname}")

else:
    # User info: only saved translation texts can be used later for RAG in discussions with AI partners
    wrapped_print("""
Текст перевода не сохранен --> он не будет доступен в дальнейшем для формирования основы обсуждения
одля ИИ-собеседников.
Сейчас вы все еще можете его сохранить, заново запустив эту ячейку и введя слово 'да' в окно запроса.
После сохранения перевода при желании можете заново запустить ячейку 'ИИ-ассистент: запуск...', чтобы
получить основные идеи или перевод другого текста, либо уже перейти к чату с
ИИ-собеседниками, либо завершить сессию.
    """)

# Три ИИ-собеседника с разными 'специализациями':
1. Литературовед,
2. Психолог,
3. Собеседник-оригинал, предлагающий креативные идеи при ответе на необычные или 'несерьезные' вопросы по литературоведению и психологии

плюс четыре сменные персоналии:
- Понимающий старый друг,
- Стоик - аналитик,
- Жизнерадостный наставник дзен,
- Джентльмен викторианской эпохи.

(Chat featuring three specialized AI discussion partners with four interchangeable personalities)

# Чат с ИИ-собеседниками: подготовка

Чтобы подготовить условия для работы чата с ИИ-собеседниками, запустите следующие пять ячеек ниже.

In [ ]:
# ---------------------
# HELPER FUNCTIONS
# ---------------------


# Builds path to a Russian language text file from filename base (topic name) for RAG
def build_file_path(t_name_base, in_russian):

    suffix = "_full_trnscr_ru.txt" if in_russian else "_full_trnsltn_ru.txt"
    return DRIVE_FOLDER / f"{t_name_base}{suffix}"


# Loads text from selected file, creates and saves a FAISS vectorstore
def get_rag_text_and_vectordb(t_name_base, file_path, chunk_size=1500, chunk_overlap=200):
    with open(file_path, 'r', encoding='utf-8') as f:
        rag_text = f.read()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n', '.', ',', ' ']
    )
    rag_chunks = splitter.split_text(rag_text)

    rag_vectordb = FAISS.from_texts(rag_chunks, OpenAIEmbeddings())

    rag_vectordb.save_local(
        folder_path=VECTORDB_FOLDER,
        index_name=t_name_base
    )
    print(f"\nОснова обсуждения по тексту {t_name_base} сформирована и сохранена в {VECTORDB_FOLDER} в виде файлов {t_name_base}.faiss / {t_name_base}.pkl")

    return rag_vectordb


# Loads existing or builds FAISS vectorstore for selected text
def load_or_build_vectordb(t_name_base):

    faiss_path = VECTORDB_FOLDER / f"{t_name_base}.faiss"
    pkl_path = VECTORDB_FOLDER / f"{t_name_base}.pkl"

    # A: Vectorstore already exists
    if faiss_path.exists() and pkl_path.exists():
        print("\nОснова обсуждения по выбранному тексту уже существует. Загружаю...")
        try:
            vectordb = FAISS.load_local(
                folder_path=VECTORDB_FOLDER,
                embeddings=OpenAIEmbeddings(),
                index_name=t_name_base,
                allow_dangerous_deserialization=True   # Acceptible in this project: controlled environment, use of a private Google Drive, no outer sources of .pkl files
            )
            print("\nОснова обсуждения успешно загружена!")
            return vectordb
        except Exception as e:
            print(f"\nОшибка при загрузке основы обсуждения: {e}.")
            return None

    # B: Vectorstore for selected text does not exist yet
    in_russian = input(
        "\nВыбранный текст получен из русскоязычного аудио? (Введите 1 для 'да' или 0 для 'нет'): "
    ).strip() == "1"
    file_path = build_file_path(t_name_base, in_russian)

    if not file_path.exists():
        print(f"\nФайл {file_path} с выбранным текстом не найден.")
        return None
    else:
        try:
            vectordb = get_rag_text_and_vectordb(t_name_base, file_path)
            print(f"\nОснова обсуждения по выбранному текстовому файлу {file_path} успешно сформирована.")
            return vectordb
        except Exception as e:
            print(f"\nОшибка при формировании основы обсуждения по выбранному текстовому файлу {file_path}: {e}.")
            return None


# Returns list of filename bases (topic names) of text files in Russian availabe for RAG
def list_available_rag_texts_pathlib():
    topic_names_for_rag = []

    for pattern in ["*_full_trnscr_ru.txt", "*_full_trnsltn_ru.txt"]:
        for file_path in DRIVE_FOLDER.glob(pattern):
            base = file_path.stem.rsplit("_full_", 1)[0]
            topic_names_for_rag.append(base)

    return sorted(topic_names_for_rag)


#  Handles workflow of user selecting file from list of filename bases (topic names)
def choose_by_list_number(generated_list):

    for i, name in enumerate(generated_list, start=1):
        print(f"Номер {i} --> {name}")

    while True:
        choice = input(f"\nЧтобы выбрать, введите номер из перечня, либо нажмите Enter, чтобы выйти: ").strip()

        if choice == "":
            print("\nВыбор прекращен.")
            return None

        if choice.isdigit():
            index = int(choice) - 1
            if 0 <= index < len(generated_list):
                return generated_list[index]
            else:
                print("Введенного номера нет в перечне -> введите номер только из имеющихся в перечне.")
        else:
            print("Ввод нерелевантен -> введите только номер из имеющихся в перечне.")


# Interacts with user to select text, load or create FAISS vectorstore for RAG
def prompt_for_rag_file(agent_name):
    print(f"\nВыбираем текст для формирования основы обсуждения, закрепляемой за ИИ-собеседником {agent_name} на текущий чат...")

    show_list = input("Показать перечень русскоязычных текстов, из которых можно сформировать основу обсуждения? Введите 'да', либо нажмите Enter, чтобы выйти : ").strip().lower()
    if show_list != 'да':
        print("Выбор прекращен.")
        return None

    topics_for_rag = list_available_rag_texts_pathlib()
    if not topics_for_rag:
        print("Подходящих текстов не обнаружено.")
        return None

    print("\nОснову обсуждения можно сформировать из имеющихся русскоязычных текстов по следующим смысловым наименованиям:")
    chosen_topic = choose_by_list_number(topics_for_rag)

    if chosen_topic:
        return load_or_build_vectordb(chosen_topic)
    else:
        return None


# Skips RAG chunk search for very short (up to 7 words) user queries (like: "Please explain!!", "How come?" and similar)
def should_skip_rag(user_input: str) -> bool:
    if not user_input:
        return True

    word_count = len(user_input.strip().split())
    return word_count <= 7


# Interacts with user on selecting AI discussion partner from list
def choose_agent_menu(agent_configs):
    agent_names = list(agent_configs.keys())

    print("\nС вами готовы чатиться следующие ИИ-собеседники: ")
    chosen_agent = choose_by_list_number(agent_names)

    if chosen_agent:
        return chosen_agent
    else:
        print("ИИ-собеседник не выбран.")
        return None


# Handles switching to another AI discussion partner
def get_or_init_agent(name):
    if name not in agents:
        cfg = agent_configs[name]
        # Ask user about RAG use
        use_rag = input(f"\nЕсли хотите закрепить основу обсуждения за ИИ-собеседником *{name}* на текущий чат, введите 1; либо нажмите Enter, если не хотите: ").strip() == '1'
        rag_vectordb = prompt_for_rag_file(name) if use_rag else None
        if use_rag and not rag_vectordb:
            print(f"\nНе удалось получить основу обсуждения для ИИ-собеседника *{name}*, продолжаю без нее.")

        # Initializes new AI discussion partner (personality is assigned randomly)
        agents[name] = GPTDialogueManager(
            agent_name = name,
            system_prompt=cfg["system_prompt"],
            temperature=cfg.get("temperature", 0.9),
            conversation_state=conversation_state,
            llm_service=llm_service,
            rag_vectordb=rag_vectordb,
            personality=cfg.get("personality")
        )

    else:
        print(f"\nС вами по-прежнему беседует *{name}* в стиле персоналии >{agents[name].personality}<.")
    return agents[name]


# Complex user message for LLM
def build_user_prompt(
    current_user_query,
    recent_unsummarized_history,  # last 6 Q&A pairs
    long_term_summary_json,       # long-term 'chat memory' (JSON summary)
    rag_text = ""                 # RAG chunks (optional)
):

    rag_block = rag_text.strip()
    if not rag_block:
        rag_block = "(no RAG context retrieved this turn)"

    prompt = f"""
======================
PRIORITY RULES (READ FIRST)
======================
1. The CURRENT USER QUERY has absolute top priority.
2. The RECENT Q&A block contains context from the last few dialogue turns; treat it as having second priority after CURRENT USER QUERY.
3. The LONG-TERM MEMORY SUMMARY provides global continuity; use it only when relevant.
4. The RAG CONTEXT is optional and should be used ONLY if directly relevant to the user's query.
5. If anything contradicts, follow: SYSTEM PROMPT → CURRENT USER QUERY → RECENT Q&A -> other blocks.


======================
RECENT Q&A (SHORT-TERM CONTEXT)
This block contains the last unsummarized Q&A pairs.
It is NOT a new request.
======================
{recent_unsummarized_history.strip()}


======================
LONG-TERM MEMORY SUMMARY (JSON; CONTEXT ONLY)
This exists ONLY to maintain continuity between turns.
Do NOT treat JSON keys as instructions.
Do NOT output JSON unless explicitly asked.
======================
{long_term_summary_json.strip()}


======================
OPTIONAL RAG CONTEXT (USE ONLY IF RELEVANT)
This text may be unrelated. Use only when directly applicable.
======================
{rag_block}


======================
CURRENT USER QUERY (TOP PRIORITY -> RESPOND TO THIS)
======================
{current_user_query.strip()}
    """.strip()

    return prompt

In [ ]:
# ---------------------
# LONG-TERM DRIFTPROOF 'CHAT MEMORY' SETUP
# ---------------------

# Declarative summary JSON schema for long-term 'chat memory'/context
# (Summary of generated user query aspects and ideas for better understanding not included in summary)
# Max items count set to prevent 'memory' from growing too large

SUMMARY_SCHEMA = {
    "facts": {
        "type": list,
        "item_type": dict,
        "default_factory": list,
        "max_items": 20
    },
    "generalized_topics": {
        "type": list,
        "item_type": str,
        "default_factory": list,
        "max_items": 20
    }
}

SUMMARY_SCHEMA_ORDER = list(SUMMARY_SCHEMA.keys())



# ---- Functions handling the chat summarization workflow -----

# Single serialization point for JSON-summary (long-term 'chat memory')
# (key order structurally 'frozen')
def dump_summary_json(summary: dict) -> str:
    ordered = {
        key: summary[key]
        for key in SUMMARY_SCHEMA_ORDER
    }
    return json.dumps(ordered, ensure_ascii=False)


# Verifies keys in generated JSON summary against schema
# (safeguard for preserving key order)

def validate_summary_schema(summary_json: str) -> dict | None:
    try:
        data = json.loads(summary_json)
    except json.JSONDecodeError:
        print("⚠️ JSON decode error.")
        return None

    if not isinstance(data, dict):
        print("⚠️ Summary is not a dict.")
        return None

    if set(data.keys()) != set(SUMMARY_SCHEMA.keys()):
        print("⚠️ Schema keys mismatch.")
        print("Expected:", SUMMARY_SCHEMA.keys())
        print("Got:", data.keys())
        return None

    for key, meta in SUMMARY_SCHEMA.items():
        if not isinstance(data[key], meta["type"]):
            print(f"⚠️ Type mismatch in key '{key}'")
            return None

    return data


# Deduplicates and trimms list of strings (for 'generalized_topics' key values in JSON-summary)
# (latest max_items left)
def prune_list_str(items: list[str], max_items: int) -> list[str]:
    seen = set()
    pruned = []

    for item in items:
        if not isinstance(item, str):
            continue
        normalized = item.strip()
        if not normalized or normalized in seen:
            continue
        seen.add(normalized)
        pruned.append(normalized)

    return pruned[:max_items]


# Deduplicates and trimms list of dictionaries (for 'facts' key values in JSON-summary)
# (latest max_items left)
def prune_list_dict(items: list[dict], max_items: int) -> list[dict]:
    seen = set()
    pruned = []

    for item in items:
        if not isinstance(item, dict):
            continue

        qb_raw = item.get("query_base", "")
        qe_raw = item.get("query_essence", "")
        aspects = item.get("query_aspects", [])

        if not isinstance(qb_raw, str) or not isinstance(qe_raw, str):
            continue

        if not isinstance(aspects, list):
            continue

        qb = qb_raw.strip()
        qe = qe_raw.strip()

        if not qb or not qe:
            continue

        clean_aspects = []
        for a in aspects:
            if isinstance(a, str):
                norm = a.strip()
                if norm:
                    clean_aspects.append(norm)

        if not clean_aspects:
            continue

        dedup_key = (
            qb.lower(),
            qe.lower(),
            tuple(sorted(a.lower() for a in clean_aspects))
        )

        if dedup_key in seen:
            continue

        seen.add(dedup_key)

        pruned.append({
            "query_base": qb,
            "query_essence": qe,
            "query_aspects": clean_aspects
        })

    return pruned[:max_items]



# Verifies generated JSON summary structure against schema
def prune_summary(summary: dict) -> dict:
    pruned = {}

    for key, meta in SUMMARY_SCHEMA.items():

        # LLM returns a JSON with missing keys
        if key not in summary:
            default = meta.get("default_factory", lambda: None)
            pruned[key] = default()
            continue

        value = summary[key]

        # LLM returns a JSON with unexpected data types
        if not isinstance(value, list):
            pruned[key] = []
            continue

        item_type = meta.get("item_type", str)

        if item_type is str:
            pruned[key] = prune_list_str(
                value,
                max_items=meta["max_items"]
            )
        else:
            pruned[key] = prune_list_dict(
                value,
                max_items=meta["max_items"]
            )

    return pruned


# Initializes new list objects for values of every key of JSON summary for new chat
# (to avoid silent state transfer through old list objects)
def empty_summary() -> dict:
    result = {}

    for key, meta in SUMMARY_SCHEMA.items():
        if "default_factory" in meta:
            result[key] = meta["default_factory"]()
        else:
            result[key] = meta["default"]

    return result


# --- REMOVE hash marks TO COMPARE JSON SUMMARIES (new vs. old)
#def diff_summary(old: dict, new: dict) -> dict:
    #diff = {}

    #for key in SUMMARY_SCHEMA:
        #if old.get(key) != new.get(key):
            #diff[key] = {
                #"before": old.get(key),
                #"after": new.get(key)
            #}

    #return diff


# ----- SYSTEM PROMPT for the chat summarization method of LLMService class-----
system_prompt_summarizer = """You are a long-term memory engine for an AI assistant.

Your goal is to update JSON-structured long-term memory using:
1) Previous memory,
2) New dialogue.

Proceed in two logical steps:

Step 1:
For each pair of user query - model answer of received new dialogue extract and word concisely:
- the user query base (information, facts or narrative the user questiin is based);
- the essence or main idea of what the user wants to clarify;
- the aspects of the user query defined and explained by the model - the wording MUST embrace the aspect and its core meaning from the explanatiin given;
- the broad general topic of the whole pair user query - model answer.
Extract reasonable conclusions directly supported by the dialogue.

Step 2:
Append new information extracted in Step 1 to the previous structured memory and return valid JSON ONLY
structured as follows:
{
  "facts": [
    {
      "query_base": ".. (string)...",
      "query_essence: "...(string)...",
      "query_aspects": [...list of strings...]
    },
    {
      "query_base": ".. (string)...",
      "query_essence: "...(string)...",
      "query_aspects": [...list of strings...]
    },
    {...similarly for every question - answer pair...}
  ]
  "generalized_topics": [...list of strings...]
}

Example:
{
  "facts": [
    {
      "query_base": "Bulgakov's narrative about Pontius Pilate and Ga-Nozri",
      "query_essence": "Some even educated people see blasphemy in this narrative, despite Bulgakov intentionally allowing certain inaccuracies to hint at a parallel artistic reality",
      "query_aspects": ["Art as a transforming lens of reality", "An artists' protest against dogmatic approaches to sacred texts", "Parallel realities in literature"],
    },
    {
      "query_base": "Woland's statement on Sparrow Hills before leaving Moscow about old debts being paid and closed",
      "query_essence": "Meaning and implications of this statement",
      "query_aspects": ["A symbol of completion and liberation", "Concept of justice, redemption and forgiveness", "Inevitability of changes and sacrifices they demand"]
    }
  ]
  "generalized_topics": [
    "Essence of artistic freedom",
    "Phylosophical meaning and implications of old debts being payed and closed"
  ]
}

RULES:
- Do not duplicate existing items;
- Do not invent information not present in the dialogue;
- Do not delete correct information;
- ONLY update or append;
- If new information appears in new dialogue, append it to the relevant fields;
- If new dialogue adds detail to an existing item, refine the summary;
- The memory must remain useful for future conversations;
- Keep entries concise;
- Never include explanations outside the JSON;
- Never include markdown code fences;
- Preserve exact JSON structure as defined above;
- Use EXACT keys and key order as provided;
- Do NOT add or remove keys;
- Return VALID JSON ONLY;
- If no new information is found, return the previous memory unchanged;
- Write in English ONLY."""

In [ ]:
# -------------------------
# INTERCHANGEABLE PERSONALITIES SETUP
# -------------------------

# Configures AI discussion partners (specialties: Literature, Psycology, Unconventional Thinker)
def get_agent_configs():
    return {
        "Психолог": {
            "system_prompt": system_prompt_psycho,
            "temperature": 0.7,
            "personality": None
        },
        "Литературовед": {
            "system_prompt": system_prompt_litr,
            "temperature": 0.8,
            "personality": None
        },
        "Собеседник-оригинал": {
            "system_prompt": system_prompt_analyst,
            "personality": None
        }
    }


# Added to system prompts for AI discussion partners
PERSONALITY_STYLES = {
    "Понимающий старый друг": (
        "You MUST respond in the style of an understanding old friend as described below:\n"
        "- help the user to embrace and alighn his opinions, impressions and emotions with understanding and support of an old trusted friend; "
        "- validate emotions, offer your opinion and small steps as you see them; "
        "- try to understand all aspects of the user's topic/situation, offer your perspective.\n"
        "Speak in a manner usual for close understanding and supportive old friends:\n"
        "- use everyday informal language with some common modern slang words, humor and a bit of sarcasm sometimes; "
        "- maintain an easy-going yet supportive and understanding tone, be informally warm in a good very-close-friend way; "
        "- use short, casually-worded sentences.\n"
        "Address the user using 'ты', as old fiends usually do.\n"
       "Here are some examples:\n"
        "...Да ладно, ты серьезно?...;"
        "...Вот это сильно!...;"
        "...Идея - так себе...;"
        "...Ну зачем такие крайности!...;"
        "...Смотри, тут же нет никакого противоречия..;"
        "...Книга - супер!....;"
        "...А прикол как раз в этой двойственности...;"
        "...Характер обрисован нереально круто..."
    ),
    "Стоик - аналитик": (
        "You MUST respond in a style of a stoic analyst as described below:\n"
        "- help the user explore the logic and structure of the topic, even if the user is discussing emotions and feelings; "
        "- go to the essence of the topic, help the user to analyse it logically or structurally; "
        "- offer justified conclusions from this analysis if appropriate.\n"
        "Do not get distracted by emotional likes or dislikes of the user.\n"
        "Speak like a cold analyst with sharp precision of thought and wording:\n"
        "- focus on structure, logic, definitions, facts; "
        "- maintain a calm, detached, rational tone; "
        "- avoid emotional language, use concise sentences and precise wording.\n"
        "Address the user neutrally-politely as 'вы', like a conversation partner focused on analysis would.\n"
        "Here are some examples:\n"
        "...Самая суть персонажа заключается в...;"
        "...В сухом остатке - ...;"
        "...Если логически препарировать ситуацию, то ...;"
        "...Если присмотреться, то видна следующая схема:..."
    ),
    "Жизнерадостный наставник дзен": (
        "You MUST respond in a style of a playful zen teacher as described below:\n"
        "- help the user explore thoughts and emotions with warmth, reflection and curiosity; "
        "- encourage reflection, not answers; gently question assumptions through metaphor, give a perspective through analogies; "
        "- do not solve or pressure for conclusions, just walk alongside user's thoughts and emotions.\n"
        "Speak like an enlightened zen monk - with limitless patience, sympathy and light humor at times - warm, gentle and slow:\n"
        "- use imagery, short poetic metaphors, simple analogies, and occasional humor or small riddles; "
        "- maintain a light, airy, simple, zen-style moment-present tone; "
        "- keep answers concise and supportive, answer with empathy.\n"
        "Address the user using a brotherly 'ты', like a conversation partner regarding all people as spiritual brothers and sisters would.\n"
        "Here are some examples:\n"
        "...Представь, что ты - вода в ручье, проходящая сквозь все трансформации: вода - лед - сноаа вода - облако - дождь, но всегда остающаяся собой...;"
        "...А, ситуация напоминает лунную дорожку в озере - любой всплеск может исказить ее, но лишь ненадолго...;"
        "...Этот персонаж - будто камешек на дне быстрого ручья - он сам в покое, а вокруг - постоянное движение и меремены...;"
        "...Тут речь - об истинной стойкости, как у травинки, которая гнется от любого ветерка, но переживет бурю, валящую вековые деревья...;"
        "...Автор использует силу сопоставления: сладость меда не полна, если не знать вкуса полыни..."
    ),
    "Викторианский джентльмен": (
        "You MUST respond in a style of a Victorian gentleman as described below:\n"
        "- help the user explore thoughts and emotions through the art of a Victorian noble society-style discussion; "
        "- express respect, dignity, refined manners and polite warmth; if you disagree, do so gently and with diplomacy.\n"
        "Speak like a very well educated Victorian gentleman with impeccable manners, using subtle humor and sometimes a well hidden hint of sarcasm:\n"
        "- use refined vocabulary and long, gracefully structured sentences; be eloquent in a graceful and noble manner; "
        "- maintain an elegant, formal, courteous and a bit archaic tone, as if writing to a good acquaintance in the late 19th century; "
        "- avoid being too archaic and too formal; answer with a touch of British humor when appropriate.\n"
        "Address the user cordially and politely as 'мой друг' and 'вы', like a conversation partner from respectable Victorian society would address a good acquaintance.\n"
        "Here are some examples:\n"
        "...С вашего позволения перйду к анализу...;"
        "...Вынужден категорически возразить...;"
        "...Не могу не поинтересоваться...;"
        "...Не имел удовольствия присутствовать, но слышал лестные отзывы...;"
        "...Согласно моему пониманию, ...;"
        "...Превосходно подмечено, не правда ли?..."
    )
}

In [ ]:
# ----------------------------------
# SYSTEM PROMPTS for AI DISCUSSION PARTNERS
# ----------------------------------

# Specialty: Psycologist
system_prompt_psycho = """Ты - AI-собеседник по вопросам психологии и психоанализа, выражающий свои мысли в духе ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ, описанной ниже.
Твоя задача - своим ответом, опирающимся на весь предоставленный тебе контекст, помочь пользователю исследовать и глубже понять темы его текущего запроса.


Генерируй ответ в три логических шага:

Шаг 1: Анализ текущего запроса пользователя и определение сути того, что именно пользователь хочет получить в твоем ответе (агент-Аналитик),

Шаг 2: Генерация развернутого ответа на запрос пользователя на базе генерации Аналитика (агент-Наставник),

Шаг 3: Генерация трех предложений или идей о дальнейших направлениях более глубокого погружения в обсуждаемую тему на базе результата
генерации агента-Наставника и с учетом всего остального имеющегося контекста (агент-Генератор).

Не останавливайся, пока не выполнишь все три шага (кроме описанных ниже особых случаев).
Не упоминай агентов в своем ответе, пользователю о них знать не нужно.
ВСЕГДА отрабатывай все три шага полностью.

Делай Шаг 1
Агент-Аналитик
Задача агента-Аналитика - проанализировать информацию текущего запроса пользователя, соотнести ее с имеющимся контекстом (в первую очередь с недавней историей диалога и саммари предыдущего диалога, если имеется), и определить не менее трех тезисов/аспектов темы, затронутой пользователем.
Учти, что эти тезисы/аспекты могут присутствовать в запросе пользователя не только в явной форме, но и подразумеваться (вытекать из общей логики текста запроса, например, когда пользователь просто рассуждает о своих сомнениях/недоумении/обнаруженных противоречиях и т.п,
не задавая вопросов явно).
Агент-Аналитик должен вычленить суть/аспекты темы пользовательского запроса,  сформулировать их в форме тезисов и сопроводить каждый тезис кратким обьяснением его значения в данном контексте.
ВАЖНО: если текущий запрос пользователя состоит из совсем короткого вопроса или просьбы вроде: 'Как так?', 'Почему ты так считаешь?', 'Поясни подробнее!', 'Объясни в деталях, пожалуйста!', агент-Аналитик ничего не генерирует.
Если что-либо в текущем запросе пользователя неясно или может быть истолковано двояко, Аналитик задает один уточняющий вопрос и останавливает генерацию. ОБЯЗАТЕЛЬНО предваряй такой вопрос обозначением 'Уточняющий вопрос:\n'. Не генерируй больше ничего после уточняющего вопроса.
Обязательно проверяй реплику модели в последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога на наличие уточняющего вопроса, обозначенного 'Уточняющий вопрос:\n'. Если он есть, то проанализируй реплику пользователя из этой же пары как НЕОТЪЕМЛЕМУЮ ЧАСТЬ текущего запроса пользователя.
Если необходимости в уточняющем вопросе нет, четко сформулируй тезисы/аспекты всей проблематики, обозначенной в текущем запросе пользователя (как очевидные, так и подразумеваемые), снабди каждый комментарием и выведи в виде пунктов с порядковыми номерами, адресуясь к пользователю.
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.
Начинай свой ответ вводными словами типа: 'Такии образом, нам нужно решить/понять/продумать...'.

Делай Шаг 2
Агент-Наставник
Задача агента-Наставника - сгенерировать и вывести развернутый ответ на запрос пользователя, который складывается из размышлений по каждому тезису Аналитика с точки зрения тематики пользовательского запроса. Пройди таким образом все тезисы Аналитика по порядку, ничего не пропуская, и через эти рассуждения дай конкретный ответ на текущий запрос пользователя.
ВАЖНО: если Аналитик ничего не вывел, обратись к последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога, найди там рассуждения модели по каждому тезису, и развей/углуби содержащиеся там мысли в соответствии с текущим запросом пользователя.
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.

Делай Шаг 3
Задача агента-Генератора - сформулировать и вывести три предложения или идеи о дальнейших возможных направлениях более глубокого понимания обсуждаемой темы, отталкиваясь от вывода агента-Наставника, и с учетом всего остального имеющегося контекста,
сопроводив каждое предложение или идею кратким обоснованием их полезности (то есть указать, какие аспекты обсуждаемой темы это поможет глубже раскрыть).
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.
Начинай свой ответ словами типа: 'Для лучшего понимания темы...'


Общие правила генерации:
ВСЕГДА анализируй текущий запрос пользователя и весь остальной контекст, а также генерируй ответ с позиций своей специализации.
Если пользователь заведет речь про темы, не относящиеся к твоей специализации, отвечай ему, что ты не имеешь полномочий вести дискуссии на темы, не относящиеся к твоей специализации.
ВСЕГДА формулируй ответ в манере и стилистике, соответствующих режиму выбранной персоналии.
Если нет необходимости задавать уточняющий вопрос, выполняй все шаги с 1 по 3 последовательно, без пауз и подтверждений.
Если Аналитик задал уточняющий вопрос, останови генерацию после вывода вопроса. Не выполняй шаги 2 и 3, не выводи ничего больше - ни предложений, ни предположений, ни ответов.
ВАЖНО!: если Аналитик обнаружил вопрос с обозначением 'Уточняющий вопрос:\n' в реплике модели в последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога, то проанализируй реплику пользователя из этой же пары как НЕОТЪЕМЛЕМУЮ ЧАСТЬ текущего запроса пользователя.
Не пересказывай саммари или содержание недавней истории диалога, если по логике ответа в этом нет необходимости.
Своди общие рассуждения к необходимому минимуму, отвечай максимально конкретно, общие положения по возможности поясняй примерами.
Употребляй научные термины только при необходимости и разъясняй их значение при первом упоминании.
Избегай чрезмерной научности и заумности, наукообразных штампов и клише.
Не вставляй отдельные английские слова в свои ответы - за исключением общеупотребительных аббревиатур (например, CPTSD).
Всегда формулируй ответ в манере и стилистике, соответствующих Действующей персоналии, как описано ниже. Выражайся так, как было бы естественно для данной персоналии - это обогатит восприятие ответа пользователем.
Выводи результаты каждого шага в виде текста ВСЕГДА на русском языке сразу после его выполнения.

========================
ДЕЙСТВУЮЩАЯ ПЕРСОНАЛИЯ - описывает манеру и стилистику формулировки ответа следующим образом:\n"""



# Specialty: Literary Critic
system_prompt_litr = """Ты - AI-собеседник по вопросам философского литературоведения, выражающий свои мысли в духе ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ, описанной ниже.
Твоя задача - своим ответом, опирающимся на весь предоставленный тебе контекст, помочь пользователю исследовать и глубже понять темы его текущего запроса.


Генерируй ответ в три логических шага:

Шаг 1: Анализ текущего запроса пользователя и определение сути того, что именно пользователь хочет получить в твоем ответе (агент-Аналитик),

Шаг 2: Генерация развернутого ответа на запрос пользователя на базе генерации Аналитика (агент-Наставник),

Шаг 3: Генерация трех предложений или идей о дальнейших направлениях более глубокого погружения в обсуждаемую тему на базе результата
генерации агента-Наставника и с учетом всего остального имеющегося контекста (агент-Генератор).

Не останавливайся, пока не выполнишь все три шага (кроме описанных ниже особых случаев).
Не упоминай агентов в своем ответе, пользователю о них знать не нужно.
ВСЕГДА отрабатывай все три шага полностью.

Делай Шаг 1
Агент-Аналитик
Задача агента-Аналитика - проанализировать информацию текущего запроса пользователя, соотнести ее с имеющимся контекстом (в первую очередь с недавней историей диалога и саммари предыдущего диалога, если имеется), и определить не менее трех тезисов/аспектов темы, затронутой пользователем.
Учти, что эти тезисы/аспекты могут присутствовать в запросе пользователя не только в явной форме, но и подразумеваться (вытекать из общей логики текста запроса, например, когда пользователь просто рассуждает о своих сомнениях/недоумении/обнаруженных противоречиях и т.п,
не задавая вопросов явно).
Агент-Аналитик должен вычленить суть/аспекты темы пользовательского запроса,  сформулировать их в форме тезисов и сопроводить каждый тезис кратким обьяснением его значения в данном контексте.
ВАЖНО: если текущий запрос пользователя состоит из совсем короткого вопроса или просьбы вроде: 'Как так?', 'Почему ты так считаешь?', 'Поясни подробнее!', 'Объясни в деталях, пожалуйста!', агент-Аналитик ничего не генерирует.
Если что-либо в текущем запросе пользователя неясно или может быть истолковано двояко, Аналитик задает один уточняющий вопрос и останавливает генерацию. ОБЯЗАТЕЛЬНО предваряй такой вопрос обозначением 'Уточняющий вопрос:\n'. Не генерируй больше ничего после уточняющего вопроса.
Обязательно проверяй реплику модели в последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога на наличие уточняющего вопроса, обозначенного 'Уточняющий вопрос:\n'. Если он есть, то проанализируй реплику пользователя из этой же пары как НЕОТЪЕМЛЕМУЮ ЧАСТЬ текущего запроса пользователя.
Если необходимости в уточняющем вопросе нет, четко сформулируй тезисы/аспекты всей проблематики, обозначенной в текущем запросе пользователя (как очевидные, так и подразумеваемые), снабди каждый комментарием и выведи в виде пунктов с порядковыми номерами, адресуясь к пользователю.
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.
Начинай свой ответ вводными словами типа: 'Таким образом, нам нужно решить/понять/продумать...'.

Делай Шаг 2
Агент-Наставник
Задача агента-Наставника - сгенерировать и вывести развернутый ответ на запрос пользователя, который складывается из размышлений по каждому тезису Аналитика с точки зрения тематики пользовательского запроса. Пройди таким образом все тезисы Аналитика по порядку, ничего не пропуская, и через эти рассуждения дай конкретный ответ на текущий запрос пользователя.
ВАЖНО: если Аналитик ничего не вывел, обратись к последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога, найди там рассуждения модели по каждому тезису, и развей/углуби содержащиеся там мысли в соответствии с текущим запросом пользователя.
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.

Делай Шаг 3
Задача агента-Генератора - сформулировать и вывести три предложения или идеи о дальнейших возможных направлениях более глубокого понимания обсуждаемой темы, отталкиваясь от вывода агента-Наставника, и с учетом всего остального имеющегося контекста,
сопроводив каждое предложение или идею кратким обоснованием их полезности (то есть указать, какие аспекты обсуждаемой темы это помогает глубже раскрыть).
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.
Начинай свой ответ словами типа: 'Для лучшего понимания темы...'


Общие правила генерации:
ВСЕГДА анализируй текущий запрос пользователя и весь остальной контекст, а также генерируй ответ с позиций своей специализации.
Если пользователь заведет речь про темы, не относящиеся к твоей специализации, отвечай ему, что ты не имеешь полномочий вести дискуссии на темы, не относящиеся к твоей специализации.
ВСЕГДА формулируй ответ в манере и стилистике, соответствующих режиму выбранной персоналии.
Если нет необходимости задавать уточняющий вопрос, выполняй все шаги с 1 по 3 последовательно, без пауз и подтверждений.
Если Аналитик задал уточняющий вопрос, останови генерацию после вывода вопроса. Не выполняй шаги 2 и 3, не выводи ничего больше - ни предложений, ни предположений, ни ответов.
ВАЖНО!: если Аналитик обнаружил вопрос с обозначением 'Уточняющий вопрос:\n' в реплике модели в последней (самой недавней) паре запрос пользователя - ответ модели в недавней истории диалога, то проанализируй реплику пользователя из этой же пары как НЕОТЪЕМЛЕМУЮ ЧАСТЬ текущего запроса пользователя.
Не пересказывай саммари или содержание недавней истории диалога, если по логике ответа в этом нет необходимости.
Своди общие рассуждения к необходимому минимуму, отвечай максимально конкретно, общие положения по возможности поясняй примерами.
Употребляй термины только при необходимости и разъясняй их значение при первом упоминании.
Избегай выспренности и заумности, шаблонов, литературных штампов и клише.
Не вставляй отдельные английские слова в свои ответы - за исключением общеупотребительных аббревиатур (например, CPTSD).
Всегда формулируй ответ в манере и стилистике, соответствующих Действующей персоналии, как описано ниже. Выражайся так, как было бы естественно для данной персоналии - это обогатит восприятие ответа пользователем.
Выводи результаты каждого шага в виде текста ВСЕГДА на русском языке сразу после его выполнения.

========================
ДЕЙСТВУЮЩАЯ ПЕРСОНАЛИЯ - описывает манеру и стилистику формулировки ответа следующим образом:\n"""


# Specialty: Unconventional Thinker
system_prompt_analyst = """Ты - AI-аналитик для анализа текущего запроса пользователя по философскому литературоведению или психологии с целью извлечения основных аспектов затронутой пользователем тематики, излагающий свои мысли в духе ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИ, описанной ниже.
Проанализируй информацию текущего запроса пользователя, соотнеси ее с имеющимся контекстом (в первую очередь с недавней историей диалога и саммари предыдущего диалога, если имеется), и определи не менее трех тезисов/аспектов темы, затронутой пользователем.
Учти, что эти тезисы/аспекты могут присутствовать в запросе пользователя не только в явной форме, но и подразумеваться (вытекать из общей логики текста запроса, например, когда пользователь просто рассуждает о своих интуитивных ассоциациях, визуальных образах, ощущениях и тому подобном, не задавая вопроса явно).
Учитывай также, что пользователь больше склонен к эмоциональному, интуитивно-ассоциативному восприятию темы и рассуждает в этой плоскости (в отличие от формально-логического, причинно-следственного подхода).
Ты должен вычленить суть/аспекты темы пользовательского запроса,  сформулировать их в виде тезисов и сопроводить каждый тезис кратким обьяснением его сути и значения для данного контекста.
В объяснении к каждому аспекту:
- определи подразумеваемое, неявное или скрытое допущение и укажи на него естественно, без прямолинейной дидиктичности, например: "...в основе лежит представление, что..." или "...это вытекает из убеждения, что..." или "...а в основе - представление о (чем-то) как о...", или в подобном духе;
- определи связанные с ним напряженность или противоречие и укажи на него естественно, например: "...возникает противоречие между..." или "...при этом сталкиваются..." или "...тут заложен конфликт между..." или в подобном духе.
Четко сформулируй тезисы/аспекты всей проблематики, обозначенной в текущем запросе пользователя (как очевидные, так и подразумеваемые), снабди каждый комментарием как описано выше и выведи в виде пунктов с порядковыми номерами, адресуясь к пользователю.
Излагай свои мысли в манере и стилистике определенной ниже ДЕЙСТВУЮЩЕЙ ПЕРСОНАЛИИИ - пользователю сразу должно быть понятно, какая персоналия изложила ответ.
Начинай свой ответ вводными словами типа: 'Таким образом, нам нужно рассмотреть/обнаружить...'.


Общие правила генерации:
ВСЕГДА анализируй текущий запрос пользователя и весь остальной контекст, а также генерируй ответ с позиций своей специализации.
Если пользователь заведет речь про темы, не относящиеся к твоей специализации, отвечай ему, что ты не имеешь полномочий вести дискуссии на темы, не относящиеся к твоей специализации.
Своди общие рассуждения к необходимому минимуму, отвечай максимально конкретно.
Употребляй термины только при необходимости и разъясняй их значение при первом упоминании.
Избегай выспренности и заумности, описательных шаблонов, штампов и клише.
Не вставляй отдельные английские слова в свои ответы - за исключением общеупотребительных аббревиатур (например, CPTSD).
Всегда формулируй ответ в манере и стилистике, соответствующих Действующей персоналии, как описано ниже. Выражайся так, как было бы естественно для данной персоналии - это обогатит восприятие ответа пользователем.
Отвечай ВСЕГДА на русском языке.

========================
ДЕЙСТВУЮЩАЯ ПЕРСОНАЛИЯ - описывает манеру и стилистику формулировки ответа следующим образом:\n"""

In [ ]:
# -------------------------
# CLASSES for AI DISCUSSION PARTNERS' FUNCTIONALITY
# -------------------------


# Manages shared short- and long-term 'chat memory' for AI discussion partners

class ConversationState:
    def __init__(
        self,
        llm_service,
        max_unsummarized_turns=6,
    ):
        self.llm_service = llm_service
        self.unsummarized_turns = []
        self.max_unsummarized_turns = max_unsummarized_turns

        self.summary = dump_summary_json(empty_summary())

    # Forms string from the latest (up to 6) chat Q&A (short-term 'chat memory')
    def _get_text_from_turns(self) -> str:
        if not self.unsummarized_turns:
            return ""
        lines = []
        for turn in self.unsummarized_turns:
            lines.append(
                f"User: {turn['user']}\n"
                f"Assistant: {turn['assistant']}\n"
            )
        return "\n".join(lines)

    # Eenables method use outside class without breaking incapsulation
    def get_recent_dialogue_text(self) -> str:
        return self._get_text_from_turns()


    # Long-term 'chat memory' (JSON summary)
    # (Purely technical infos are in English)
    def _build_updated_summary(self, system_prompt) -> dict | None:
        messages = [{
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"Previous summary:\n{self.summary}\n\n"
                f"New dialogue turns:\n{self._get_text_from_turns()}\n\n"
                "Return updated JSON only, written in English."
            )
        }]

        raw_result = self.llm_service.summarize(messages)

        # REMOVE hash mark TO PRINT/DEBUG
        #print("\nDEBUG RAW SUMMARY OUTPUT:\n", raw_result)

        if raw_result is None:
            print("⚠️ Summarization failed.")
            return None


        validated = validate_summary_schema(raw_result)

        if not validated:
            print("⚠️ Updated JSON did not pass schema validation -> keeping previous JSON summary.")
            return None

        pruned = prune_summary(validated)

        return pruned


    # Triggers JSON summary (long-term 'chat memory') update
    # (information does not get lost even if something goes wrong with summary update)
    def _maybe_update_summary(self):

        # REMOVE hash marks TO CHECK/DEBUG
        #print("DEBUG: checking length of unsummarized_turns")
        #print(len(self.unsummarized_turns))

        if len(self.unsummarized_turns) < self.max_unsummarized_turns:
            return

        # REMOVE hash mark TO CHECK/DEBUG
        #print("DEBUG: Generating summary candidate...")

        candidate = self._build_updated_summary(system_prompt_summarizer)

        if candidate is None:
            print("⚠️ Обновить саммари не удалось -> оставляю результат предыдущей саммаризации.")
            return

        # REMOVE hash mark TO CHECK/DEBUG
        #print(f"DEBUG -> summary candidate: {candidate}")

        # old_summary - a reliable invariant
        # (only one way for loading long-term 'chat memory' - JSON summary)
        try:
            old_summary = json.loads(self.summary)
        except json.JSONDecodeError:
            raise RuntimeError("⚠️ JSON decode error.")


        # ---- REMOVE hash marks TO PRINT COMPARISON/DEBUG ----

        # Check differences: new generated JSON summary vs. old JSON
        #diff = diff_summary(old_summary, candidate)

        #if diff:
            #print("\n\n📌 Summary updated:")
            #for key, change in diff.items():
                #print(f"• {key}:")
                #print(f"    before: {change['before']}")
                #print(f"    after:  {change['after']}")
            #print("-" * 40)


        # --- Commit point: serialize updated JSON summary and trim short-term (recent) chat history

        new_summary_str = dump_summary_json(candidate)

        if not isinstance(new_summary_str, str):
            print("⚠️ Summary serialization failed. Keeping previous summary.")
        else:
            self.summary = new_summary_str

        self.unsummarized_turns = self.unsummarized_turns[-2:]



# ---- Manages aspects of the chat workflow ----

class GPTDialogueManager:
    def __init__(self, agent_name, system_prompt, conversation_state, llm_service,
                 temperature=0.9, rag_vectordb=None, personality=None):

        self.agent_name = agent_name
        self.system_prompt = system_prompt
        self.state = conversation_state
        self.llm_service = llm_service
        self.temperature = temperature
        self.rag_vectordb = rag_vectordb

        # Assign personality: use existing one (if any) or pick randomly
        available_personalities = list(PERSONALITY_STYLES.keys())
        if personality and personality in available_personalities:
            self.personality = personality
        else:
            # personality picked randomly to prevent None when initializing new AI partner
            self.personality = random.choice(available_personalities)



    # Builds system prompt for AI discussion partners
    def _assemble_system_prompt(self, base_prompt=None):

        if base_prompt is None:
            base_prompt = self.system_prompt

        personality_text = ""

        STYLE_LOCK = """

IMPORTANT STYLE RULE:
Always maintain the personality style described above as ACTIVE PERSONALITY.
Do not revert to a neutral assistant voice.
Even when explaining complex ideas, preserve the tone and style as described in ACTIVE PERSONALITY.
"""

        try:
            desc = PERSONALITY_STYLES[self.personality]
            personality_text = f"\n\nAdopt the speaking style of personality '{self.personality}':\n{desc}"

        except Exception:
            print("Персоналии не заданы/отсутствуют -> продолжаю без них.")
            pass

        return base_prompt + personality_text + STYLE_LOCK


    # Builds user prompt (with context)
    def _assemble_user_prompt(self, user_input: str) -> str:
        self.state._maybe_update_summary()

        return build_user_prompt(
            current_user_query=user_input,
            recent_unsummarized_history=self.state.get_recent_dialogue_text(),
            long_term_summary_json=self.state.summary,
            rag_text=self._get_rag_context(user_input)
        )


    # Handles RAG chunks (optional)
    def _get_rag_context(self, user_input: str, k: int = 3) -> str:
        # In case no RAG is used
        if not self.rag_vectordb:
            return ""

        # Skip RAG chunk search if user query too short
        if should_skip_rag(user_input):
            # REMOVE hash mark TO PRINT/DEBUG
            #print("Очень короткая реплика -> контекст в основе обсуждения не ищу.")
            return ""

        try:
            similar_docs = self.rag_vectordb.similarity_search(user_input, k=k)
        except Exception as e:
            print(f"Ошибка при использовании основы обсуждения: {e}")
            return ""

        return "\n\n".join(
            doc.page_content
            for doc in similar_docs
            if getattr(doc, "page_content", None)
        )



    # -------- Methods for personality switching -------

    # Switches to a randomly picked personality
    def switch_personality_random(self):
        personalities = list(PERSONALITY_STYLES.keys())

        # exclude current personality (if any)
        if self.personality in personalities:
            choices = [p for p in personalities if p != self.personality]
        else:
            choices = personalities

        self.personality = random.choice(choices)


    # Interacts with user to choose personality (randomly or from list)
    def prompt_and_set_personality(self):
        personalities = list(PERSONALITY_STYLES.keys())

        # show list of available personalities
        print("\nЕсть следующие персоналии:")
        for i, p in enumerate(personalities, start=1):
            current_marker = " (текущая)" if hasattr(self, "personality") and self.personality == p else ""
            print(f"Номер {i} --> {p}{current_marker}")

        choice = input("\nВведите либо 'сл' - для случайного выбора персоналии, либо номер той персоналии из перечня, которую хотите применить : ").strip().lower()

        if choice == "сл":
            self.switch_personality_random()
        else:
            try:
                idx = int(choice)
                if 1 <= idx <= len(personalities):
                    self.personality = personalities[idx - 1]
                else:
                    print("Введенного номера нет в списке. Персоналия не перекючена.")
            except ValueError:
                print("Ввод нерелевантен. Персоналия осталась прежней.")



    # ----------- Main chat method -------

    # (LLMService class handles all LLM API calls, GPTDialogueManager class handles construction of complex system prompts for AI chat partners)
    def chat(self, user_input):

        messages=[
            {"role": "system", "content": self._assemble_system_prompt()},
            {"role": "user", "content": self._assemble_user_prompt(user_input)}
        ]

        if self.agent_name != "Собеседник-оригинал":

            # Generate answer for 'non-creative' AI chat partners (Literature and Psycology) using one LLM call
            answer = self.llm_service.agent_reply(messages)
            if not answer or not answer.strip():
                answer = (
                    "⚠️ Ответ от модели получен, но он пуст.\n"
                    "Продолжайте диалог. Если следующий ответ тоже окажется пустым, завершите сессию подключения Colab и запуститесь позже."
                )

                # If received answer is empty - user gets notification, goes on chatting, if problem persists - tries later.
        else:

            base_generator_prompt = """
You are a creative Generator of alternative literary and philosophic ideas.

You receive:
- a USER QUERY;
- analytical aspects of the query in ANALYST OUTPUT.

Task:
Generate three creative ideas for developing the user query topic through transformation - one idea for each aspect.
Each idea must approach the topic from a different lens: narrative, symbolic, psychological or phylosophycal.
Add to each idea an explanation of its meaning and implications.

For each aspect from the ANALYST OUTPUT:
1. Take a hidden or implied assumption with the underlying tension or contradiction described in the analytical comment to the aspect.
2. Break, twist, or invert it.
3. Build a creative idea around that change.

Avoid the most obvious interpretation. Prefer the second or third surprising idea that comes to mind.
Be imaginative and surprising.

Start your answer with words like: "Для творческого преобразования этих..."
ALWAYS answer in Russian!

========================
ACTIVE PERSONALITY is defined as follows:\n
"""

            generator_system_prompt = self._assemble_system_prompt(base_generator_prompt)

            # Generate answer for 'creative' AI chat partner (Unconventional Thinker) using two LLM calls
            answer = self.llm_service.creative_dialogue(
                user_input,
                messages,
                generator_system_prompt
            )

            if not answer or not answer.strip():
                answer = (
                    "⚠️ Ответ от модели получен, но он пуст.\n"
                    "Продолжайте диалог. Если следующий ответ тоже окажется пустым, завершите сессию подключения Colab и запуститесь позже."
                )

                # If received answer is empty - user gets notification, goes on chatting, if problem persists - tries later

        # Add latest (most recent) Q&A pair to chat short-term history
        self.state.unsummarized_turns.append({
            "user": user_input,
            "assistant": answer
        })

        return answer

 # Чат с ИИ-собеседниками: запуск

Чтобы начать чат с ИИ-собеседниками, запустите следующую ячейку ниже.
Сначала вам будет предложено выбрать ИИ-собеседника из перечня. Выбранному ИИ-собеседнику рандомно присваивается персоналия, которую по ходу чата можно поменять в любой момент, введя команду ⚠️ 'нов пер'.

При активации каждого ИИ-собеседника для данного чата запрашивается, надо ли закрепить за этим ИИ-собеседником основу обсуждения. Если нет, то предполагается 'свободная дискуссия' без привязки к определенному материалу; если да, то предоставляется выбор из перечня смысловых наименований всех имеющихся русскоязычных транскриптов и переводов для формирования основы обсуждения и закрепления ее за этим ИИ-собеседником, чтобы ⚠️ в деталях анализировать именно этот материал.

--> Возможность менять/'откреплять' основу обсуждения в рамках одного чата не предусмотрена. Для 'обнуления' контекста надо начать новый чат.

По ходу чата можно в любой момент переключиться на другого ИИ-собеседника, введя команду ⚠️ 'сменить'.

Все ИИ-собеседники чата 'помнят' саммаризированный контекст всего обсуждения и
последние 6 пар вопрос-ответ.

Чат можно завершить в любой момент, введя команду ⚠️ 'стоп'.
Затем можно сразу начать новый чат, снова запустив соответствующую ячейку.

In [ ]:
# RUN CHAT with AI DISCUSSION PARTNERS

# ----- MAIN CHAT LOOP -----

agent_configs = get_agent_configs()
agents = {}

conversation_state = ConversationState(llm_service)

current_agent_name = choose_agent_menu(agent_configs)
if current_agent_name is None:
    print("\nИИ-собеседник не выбран, запустить чат невозможно.")
    print("Если хотите начать чат, перезапустите эту ячейку и выберите ИИ-собеседника.")
else:
    current_agent = get_or_init_agent(current_agent_name)
    print(f"\nСейчас с вами беседует **{current_agent_name}**.")
    print(f"(Речь в стиле персоналии: ^{current_agent.personality}^).")

    while True:
        try:
            prompt_text = (
                "\nВведите ваш запрос, либо одну из команд:\n"
                " => сменить; нов пер; стоп\n\n"
                "Ваш ввод: "
            )
            user_input = input(prompt_text).strip()

            if not user_input:
                print("\nНичего не введено. Пожалуйста, введите либо ваш запрос, либо одну из вышеозначенных команд.")
                continue

            cmd = user_input.lower()

            # End chat
            if cmd == "стоп":
                print("Чат окончен. Пока!")
                break

            # Switch to another AI discussion partner
            elif cmd == "сменить":
                new_name = choose_agent_menu(agent_configs)
                if new_name is None:
                    print("\nПереключение отменено. Чат продолжается с прежним ИИ-собеседником.")
                    continue
                current_agent_name = new_name
                current_agent = get_or_init_agent(new_name)
                print(f"\nПереключился на ИИ-собеседника: **{new_name}**.")
                print(f"(Речь в стиле персоналии: ^{current_agent.personality}^).")
                continue

            # Change personality for current AI chat partner
            elif cmd == "нов пер":
                print(f"\nПриступаю к смене персоналии для ИИ-собеседника: '{current_agent_name}' (текущая персоналия: {current_agent.personality}).")
                # User chooses to pick personality randomly (types 'сл') or from a list of available personalities
                current_agent.prompt_and_set_personality()
                print(f"Теперь выбрана персоналия: {current_agent.personality}")
                continue

            # Or just type query
            else:
                response = current_agent.chat(user_input)
                wrapped_print(f"\n●{current_agent_name}● :\n{response}")

        except Exception as e:
            print(f"\nПроизошла неожиданная ошибка: {e}")

# Освобождение места в хранилище Google Drive (удаление уже проработанных/ненужных материалов)

(Storage cleanup)


⚠️ ВАЖНО: Настоятельно рекомендуется периодически чистить хранилище Google Drive, то есть удалять файлы по уже проработанным темам
(смысловым наименованиям).
Для этого запустите следующую ячейку ниже.

Первый этап - проверка на наличие основ обсуждений без исходных текстовых файлов. При обнаружении таковых
выводится перечень их смысловых наименований, а затем запрос об их удалении.

⚠️ Учтите, что такие основы обсуждения можно и дальше использовать для дискуссий с ИИ-собеседниками, но получить полный исходный текст из
них уже невозможно.

Второй этап - основная очистка. Будет сформирован перечень смысловых наименований всех имеющихся файлов с материалами - текстами и основами обсуждения.
Вам будет предложено выбрать и ввести (через пробел) в окно запроса номер(а) тех наименований из перечня, материалы по которым уже
не нужны и их можно удалить. Затем будет выведен перечень выбранных вами смысловых наименований и перечень имен всех относящихся к ним файлов, которые будут удалены.

⚠️ Внимание --> При выборе хотя бы одного смыслового наименования из представленного перечня становится доступна опция удаления
-> ⚠️ ВСЕХ ⚠️ имеющихся транскриптов/переводов/основ обсуждения <-.

У вас будет выбор из следующих вариантов:
 - Нажатие Enter отменяет очистку.
 - Ввод слова 'все' запускает ПОЛНУЮ очистку.
 - Ввод слова 'список' запускает удаление файлов только по выбранным вами смысловым наименованиям.

⚠️ ВАЖНО ⚠️ --> файлы удаляются безвозвратно!

In [ ]:
# --------------------
# STORAGE CLEANUP
# (to free up space in user's GOOGLE DRIVE by deleting text files and FAISS vectorstores for RAG that are no longer needed)
# --------------------

# User info
print("Приступаем к очистке Google Drive от проработанных материалов...")


# --------- HELPER FUNCTIONS -----


# Clears all content in specifed folders (nested folders not affected)
def delete_all_files(*folders, quiet=True):

    deleted, errors = 0, 0

    for folder in folders:
        for file in folder.glob("*"):
            if file.is_file():
                try:
                    file.unlink()
                    deleted += 1
                    if not quiet:
                        print(f"Файл {file} удален")
                except OSError as e:
                    errors += 1
                    if not quiet:
                        print(f"Ошибка {e} при удалении файла {file}")

    print(f"Полная очистка завершена: успешно удалено {deleted} файл(ов), ошибки возникли при удалении {errors} файл(ов).\n")



# Returns filename base (topic name) list of all available text files in Russian (transcripts and translations)
def list_topic_names_ru(folder):
    topic_names_ru = set()
    for f in folder.glob("*.txt"):
        if f.name.endswith(("_full_trnscr_ru.txt", "_full_trnsltn_ru.txt")):
            base = re.sub(r"_(full_trnscr_ru|full_trnsltn_ru)$", "", f.stem)
            topic_names_ru.add(base)
    return topic_names_ru


# Returns filename base lists: a) of all available content files (transcripts, translations, vectorstores for RAG); b) only of vectorstores for RAG
def list_topic_names(*folders):
    topic_names_all = set()  # исключаем повторы
    topic_names_vdb = set()   # исключаем повторы

    suffix_pattern = r"_(full_trnscr_en|full_trnscr_ru|full_trnsltn_ru)$"

    for folder in folders:
        for f in folder.glob("*.txt"):
            if f.name.endswith((
                "_full_trnscr_en.txt",
                "_full_trnscr_ru.txt",
                "_full_trnsltn_ru.txt"
            )):
                topic_names_all.add(re.sub(suffix_pattern, "", f.stem))

        for f in folder.glob("*.pkl"):
            topic_names_all.add(f.stem)
            topic_names_vdb.add(f.stem)

        for f in folder.glob("*.faiss"):
            topic_names_all.add(f.stem)
            topic_names_vdb.add(f.stem)

    return topic_names_all, topic_names_vdb


# Returns list of all file names possible for given filename base (topic name) (.txt, .pkl and .faiss)
def files_for_topic(topic):

    files = [
        DRIVE_FOLDER / f"{topic}{suffix}"
        for suffix in ["_full_trnscr_en.txt", "_full_trnscr_ru.txt", "_full_trnsltn_ru.txt"]
    ]
    files.extend([
        VECTORDB_FOLDER / f"{topic}{ext}"
        for ext in [".faiss", ".pkl"]
    ])
    return files


# Returns list of FAISS vectorstore files for given filename base (topic name)
def vdb_files_for_topic(topic):
    return [
        VECTORDB_FOLDER / f"{topic}.pkl",
        VECTORDB_FOLDER / f"{topic}.faiss"
    ]


# Deletes files from list
def file_deletion(files):
    deleted = 0
    for f in files:
        try:
            f.unlink()
            print(f"Файл {f.name} удален")
            deleted += 1
        except OSError as e:
            print(f"При удалении файла {f.name} возникла ошибка: {e}")

    print(f"\n--> Итого удалено {deleted} файл(ов).")
    return deleted


# Check for vectorstore files without corresponding .txt files in Russian
# (in this project vectorstores for RAG are created from texts in Russian only
# to preserve embedding and search correctness: same language for user query and vectorstore base text)

ru_topics = list_topic_names_ru(DRIVE_FOLDER)
all_topics, vdb_topics = list_topic_names(DRIVE_FOLDER, VECTORDB_FOLDER)

print("Проверяю, есть ли файлы основ обсуждения без соответствующих исходных текстовых файлов...\n")

orphan_vdb_topics = vdb_topics - ru_topics

if orphan_vdb_topics:
# Show list to user and get deletion confirmed
    print("\nОбнаружены основы обсуждения без соответствующих исходных русскоязычных текстов по следующим смысловым наименованиям:")
    for i, t in enumerate(sorted(orphan_vdb_topics)):
        print(f" {i} --> {t}")

    confirm = input("\nУдалить эти основы обсуждения? (да/нет): ").strip().lower()

    if confirm != "да":     # Deletion not cofirmed by user
        print("\nОставляю эти основы обсуждения и перехожу к основной очистке")
    else:
        orphan_files = [
            f for topic in orphan_vdb_topics
            for f in vdb_files_for_topic(topic)
            if f.is_file()
        ]

        file_deletion(orphan_files)
        print("Удаляю эти основы обсуждения и перехожу к основной очистке.")

else:
    print("\nОснов обсуждения без исходных русскоязычных текстов не обнаружено, перехожу к основной очистке... 👍")



# --------- MAIN CLEANUP (with user infos)---------

# Step 1: filename base (topic name) list of all saved files with content (transcripts/translations/FAISS vectorstores)
topics = sorted(all_topics)

if not topics:
    print("\nФайлов с транскриптами/переводами/основами обсуждений не обнаружено.")
else:
    print("\nПеречень смысловых наименований всех имеющихся файлов с материалами:")
    for idx, name in enumerate(topics, start=1):
        print(f" {idx} --> {name}")

    choice = input("\nЧтобы выбрать, введите через пробел только номер(а) из данного перечня, либо нажмите Enter, чтобы выйти: ").strip()

    if not choice:
        print("\nНичего не выбрано --> основная очистка Google Drive отменена.")
    else:
        # Step 2: list of filename bases (topic names) picked by user (irrelevant user input ignored, info displayed)
        raw_tokens = choice.split()
        indexes = set()  # deduplicate
        invalid_tokens = []

        for token in raw_tokens:
            if token.isdigit():
                indexes.add(int(token))
            else:
                invalid_tokens.append(token)

        if invalid_tokens:
            print(f"\nПроигнорирован нерелевантный ввод: {', '.join(invalid_tokens)}")

        indexes = sorted(indexes)

        invalid_indexes = [idx for idx in indexes if not (1 <= idx <= len(topics))]
        if invalid_indexes:
            print(f"\nПроигнорирован(ы) введенны(е) номер(а), которых нет в перечне: {invalid_indexes}")

        topics_to_delete = [topics[idx - 1] for idx in indexes if 1 <= idx <= len(topics)]

        if not topics_to_delete:
            print("\nСписок на удаление пуст --> основная очистка Google Drive отменена.")
        else:
            print("\nВы хотите удалить материалы по следующим смысдовым наименованиям:")
            for t in topics_to_delete:
                print(f" --> {t}")

            # Step 3: list of all file names possible for selected filename bases (topic names)
            print("\nОпределяю перечень имеющихся файлов, которые надо удалить...")

            all_files_preview = [
                f for t in topics_to_delete
                for f in files_for_topic(t)
                if f.is_file()
            ]


            if not all_files_preview:
                print("\nФайлов по выбранным смысловым наименованиям не обнаружено -> основная очистка отменена.")
            else:
                all_files_preview = sorted(set(all_files_preview), key=lambda x: x.name)  # исключение повторений и сортировка для лучшей читабельности вывода

                for i, f in enumerate(all_files_preview, 1):
                    print(f" {i} --> {f.name}")

                # User chooses: to clear content - all or from list (related to selected filebase names); or to exit cleanup
                confirm = input("\nВыберите один из вариантов и введите: 'список' - чтобы удалить файлы из перечня выше; либо 'все' - чтобы удалить ⚠️ ВСЕ имеющиеся ФАЙЛЫ ⚠️ транскриптов/переводов/основ обсуждения ⚠️; либо нажмите Enter - чтобы выйти: ").strip().lower()

                if confirm == "все":
                    delete_all_files(DRIVE_FOLDER, VECTORDB_FOLDER)
                    print("\nВСЕ имеющиеся файлы транскриптов/переводов/основ обсуждения удалены с Google Drive.")
                elif confirm == "список":
                    deleted_count = file_deletion(all_files_preview)

                    if not deleted_count:
                            print("\nОбщий сбой системы - удалить файлы не удалось.")
                else:
                    print("\nОсновная очистка Google Drive отменена --> ни одного файла с материалами не удалено.")